# Seeing past the net: Inrange ball-flight prediction

Predict launch spin, apex and level landing for golf shots seen only up to a 60 m net, then show the whole flight from launch to rest.

**Approach.** A 3-D flight model (drag, Magnus lift, spin decay, session wind) is calibrated on the training shots and inverted on each shot's four checkpoints. Its prediction becomes a feature for a Gaussian process per target, and every prediction is turned back into a physically consistent trajectory with bounce and roll.

| 5-fold CV on train (all refits inside each fold) | landing | apex | apex time | landing time | spin |
|---|---|---|---|---|---|
| final hybrid | 4.24 m | 2.09 m | 0.068 s | 0.118 s | 718 rpm |

**This notebook:**
1. writes the project's source modules
2. calibrates the aerodynamics
3. runs honest cross-validation and writes `submission.csv`
4. analyses the errors (what, why and when they happen)
5. renders the report figures
6. animates a test shot

It runs in roughly 15 minutes on a Kaggle CPU session. The Kaggle Writeup has the full story.

## Setup
Finds the competition data (Kaggle input or a local copy) and prepares the working folders.

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

WORK = Path.cwd()
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)]
candidates += [WORK / "inrange-competition", WORK.parent / "inrange-competition"]
data_dir = next((p for p in candidates if (p / "train.csv").exists() and (p / "test.csv").exists()), None)
assert data_dir is not None, "Competition data not found: add the Inrange competition data to this notebook."
os.environ["INRANGE_DATA_DIR"] = str(data_dir)
os.environ["PYTHONPATH"] = str(WORK / "src")
for sub in ["src/data", "src/flight", "src/modelling", "src/experiments", "src/report", "outputs", "figures", "submissions"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK / "src"))


def run(*args, tail=25):
    """Run a project module the way the repository does (python -m ...) and show the end of its output."""
    env = {**os.environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
    result = subprocess.run([sys.executable, "-m", *args], capture_output=True, text=True, env=env, cwd=WORK)
    lines = [line for line in result.stdout.splitlines() if "warn" not in line.lower()]
    print("\n".join(lines[-tail:]))
    if result.returncode:
        print(result.stderr[-4000:])
        raise RuntimeError(f"python -m {' '.join(args)} failed")


print("data:", data_dir)

## Source code
These cells write the project modules exactly as they are in the repository, grouped by purpose:
- **data:** loading, coordinate frames and features
- **flight:** simulator, calibration, inversion and ground contact
- **modelling:** evaluation and the final pipeline
- **experiments:** the studies behind the model choice and the error analysis
- **report:** figures and animation

### `src/data/dataset.py`
Range geometry recovered from the data, per-shot launch frame, hitting blocks, submission format.

In [ ]:
%%writefile src/data/dataset.py
"""Competition data: loading, range and per-shot coordinate frames, hitting blocks and submission format."""
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = Path(os.environ.get("INRANGE_DATA_DIR", ROOT / "inrange-competition"))

# Downrange heading recovered from the checkpoint lines: every cp_k crossing projects onto this
# direction at exactly the same value no matter which bay the shot was hit from.
HEADING = 0.4950901162353211                      # rad from +x (28.367 deg)
U = np.array([np.cos(HEADING), np.sin(HEADING)])  # downrange unit vector
V = np.array([-U[1], U[0]])                       # lateral unit vector, + = left of target line
CP_PLANES = 10.8621 + 15.0 * np.arange(4)         # range-frame downrange coordinate of the lines

POINTS = ["cp1", "cp2", "cp3", "cp4"]
TARGETS = ["launch_spin_rate", "apex_t", "apex_x", "apex_y", "apex_z",
           "landing_t", "landing_x", "landing_y", "landing_z"]
LOCAL_TARGETS = ["launch_spin_rate", "apex_t", "apex_d", "apex_l", "apex_h",
                 "landing_t", "landing_d", "landing_l"]


def load():
    train = add_local(pd.read_csv(DATA_DIR / "train.csv"))
    test = add_local(pd.read_csv(DATA_DIR / "test.csv"))
    add_blocks(train, test)
    return train, test


def add_local(df):
    """Add launch-relative downrange (d), lateral (l) and height (h) columns plus launch angles."""
    df = df.copy()
    lxy = df[["launch_x", "launch_y"]].to_numpy()
    for p in POINTS + ["apex", "landing"]:
        if f"{p}_x" not in df:
            continue
        rel = df[[f"{p}_x", f"{p}_y"]].to_numpy() - lxy
        df[f"{p}_d"] = rel @ U
        df[f"{p}_l"] = rel @ V
        df[f"{p}_h"] = df[f"{p}_z"] - df["launch_z"]
    vxy = df[["launch_vx", "launch_vy"]].to_numpy()
    df["vd"] = vxy @ U
    df["vl"] = vxy @ V
    df["vh"] = df["launch_vz"]
    df["speed"] = np.sqrt(df.vd**2 + df.vl**2 + df.vh**2)
    df["vla"] = np.degrees(np.arcsin(df.vh / df.speed))
    df["hla"] = np.degrees(np.arctan2(df.vl, df.vd))
    df["tee"] = df.launch_x.round(2).astype(str) + "_" + df.launch_y.round(2).astype(str)
    df["elevated"] = (df.launch_z > 1).astype(int)
    df["session"] = pd.to_datetime(df.launch_time, unit="s").dt.strftime("%Y-%m-%d")
    return df


def add_blocks(train, test, gap=900.0):
    """Label contiguous hitting blocks (no pause longer than `gap` s) on the joint timeline."""
    both = pd.concat([train[["track_id", "launch_time"]], test[["track_id", "launch_time"]]])
    both = both.sort_values("launch_time")
    both["block"] = (both.launch_time.diff() > gap).cumsum()
    block = both.set_index("track_id").block
    for df in (train, test):
        df["block"] = df.track_id.map(block).to_numpy()


def to_global(df, d, l, h):
    """Map launch-relative (d, l, h) back to range x, y, z for each row of df."""
    x = df.launch_x.to_numpy() + d * U[0] + l * V[0]
    y = df.launch_y.to_numpy() + d * U[1] + l * V[1]
    z = df.launch_z.to_numpy() + h
    return x, y, z


def to_submission(df, pred):
    """Convert a local-frame prediction frame (LOCAL_TARGETS columns) to the submission format."""
    out = pd.DataFrame({"track_id": df.track_id.to_numpy()})
    out["launch_spin_rate"] = pred["launch_spin_rate"].to_numpy()
    for p in ["apex", "landing"]:
        h = pred[f"{p}_h"].to_numpy() if f"{p}_h" in pred else np.zeros(len(df))
        x, y, z = to_global(df, pred[f"{p}_d"].to_numpy(), pred[f"{p}_l"].to_numpy(), h)
        out[f"{p}_t"] = pred[f"{p}_t"].to_numpy()
        out[f"{p}_x"], out[f"{p}_y"], out[f"{p}_z"] = x, y, z
    return out[["track_id"] + TARGETS]


### `src/data/features.py`
Kinematic fingerprint features: effective lift and drag coefficients measured between checkpoints.

In [ ]:
%%writefile src/data/features.py
"""Physically motivated features from the launch and the four checkpoint crossings."""
import numpy as np
import pandas as pd

from data.dataset import POINTS
from flight.simulator import AREA, G, MASS

K_NOMINAL = 1.19 * AREA / (2 * MASS)


def kinematic_features(df):
    """Physically motivated summaries of the visible 60 m of flight.

    Each axis is fitted with r(t) = v0 t + a t^2/2 + j t^3/6 through the four checkpoints, anchored
    at the radar launch velocity v0. The implied aerodynamic acceleration (total minus gravity) is
    split into drag along the velocity and lift across it (vertical-plane and sideways parts),
    normalised by the dynamic pressure so they read as effective drag and lift coefficients.
    """
    t = df[[f"{p}_t" for p in POINTS]].to_numpy()
    r = np.stack([df[[f"{p}_{c}" for p in POINTS]].to_numpy() for c in "dlh"], axis=-1)
    v0 = df[["vd", "vl", "vh"]].to_numpy()
    basis = np.stack([t**2 / 2, t**3 / 6], axis=-1)
    lhs = np.einsum("nki,nkj->nij", basis, basis)
    rhs = np.einsum("nki,nkc->nci", basis, r - t[..., None] * v0[:, None, :])
    coef = np.linalg.solve(lhs[:, None], rhs[..., None])[..., 0]  # (N, axis, [a, j])
    resid = r - t[..., None] * v0[:, None, :] - np.einsum("nki,nci->nkc", basis, coef)

    feats = {"fit_rms": np.sqrt((resid**2).mean(axis=(1, 2)))}
    for label, tt in [("mid", t[:, 1]), ("end", t[:, 3])]:
        vel = v0 + coef[..., 0] * tt[:, None] + coef[..., 1] * tt[:, None] ** 2 / 2
        aero = coef[..., 0] + coef[..., 1] * tt[:, None]
        aero[:, 2] += G
        speed = np.linalg.norm(vel, axis=1)
        vhat = vel / speed[:, None]
        q = K_NOMINAL * speed**2
        a_par = (aero * vhat).sum(1)
        perp = aero - a_par[:, None] * vhat
        up = np.array([0.0, 0.0, 1.0]) - vhat[:, 2:3] * vhat
        up /= np.linalg.norm(up, axis=1, keepdims=True)
        side = np.cross(up, vhat)  # + = left
        feats[f"cd_{label}"] = -a_par / q
        feats[f"clu_{label}"] = (perp * up).sum(1) / q
        feats[f"cls_{label}"] = (perp * side).sum(1) / q
        feats[f"v_{label}"] = speed
        feats[f"vh_{label}"] = vel[:, 2]
        feats[f"vl_{label}"] = vel[:, 1]
    t4 = t[:, 3]
    feats["lift_proxy"] = r[:, 3, 2] - (v0[:, 2] * t4 - G * t4**2 / 2)
    feats["time_excess"] = t4 - r[:, 3, 0] / v0[:, 0]
    feats["side_excess"] = r[:, 3, 1] - v0[:, 1] * t4
    return pd.DataFrame(feats, index=df.index)


BASE_FEATURES = ["vd", "vl", "vh", "speed", "vla", "hla", "elevated", "cp1_d"] + [
    f"{p}_{c}" for p in POINTS for c in "tlh"]


def feature_frame(df):
    return pd.concat([df[BASE_FEATURES], kinematic_features(df)], axis=1)


### `src/flight/simulator.py`
Batched RK4 flight model: gravity, drag, Magnus lift, spin decay and wind.

In [ ]:
%%writefile src/flight/simulator.py
"""Batched point-mass golf-ball flight model: gravity, drag, Magnus lift, spin decay and wind.

Every shot is integrated at once with RK4 in its own launch-relative frame
(d = downrange, l = lateral with + to the left, h = height above the tee), so a few
thousand trajectories cost a handful of numpy operations per time step.
"""
import numpy as np

G = 9.81
MASS = 0.04593          # kg, USGA maximum
RADIUS = 0.021335       # m, USGA minimum diameter / 2
AREA = np.pi * RADIUS**2
RPM_TO_RAD = 2 * np.pi / 60

# Starting aerodynamic model; the free coefficients are calibrated on the training shots.
DEFAULT_AERO = dict(
    rho=1.19,                    # kg/m^3, ~120 m ASL at ~20 C
    cd0=0.20, cd1=0.25,          # CD = cd0 + cd1 * S
    cl_max=0.45, s_half=0.12,    # CL = cl_max * S / (S + s_half)
    decay=1.0e-3,                # d(omega)/dt = -decay * omega * |v_air|   [1/m]
)


def coefficients(S, aero):
    cd = aero["cd0"] + aero["cd1"] * S
    cl = aero["cl_max"] * S / (S + aero["s_half"])
    return cd, cl


def spin_axis(v0, tilt):
    """Unit spin axis perpendicular to the launch velocity; tilt > 0 curves the ball left."""
    vhat = v0 / np.linalg.norm(v0, axis=1, keepdims=True)
    side = np.stack([vhat[:, 1], -vhat[:, 0], np.zeros(len(v0))], axis=1)  # horizontal, to the right
    side /= np.linalg.norm(side, axis=1, keepdims=True)
    up = np.cross(side, vhat)
    return np.cos(tilt)[:, None] * side + np.sin(tilt)[:, None] * up


def _accel(vel, om, axis, wind, aero):
    va = vel - wind
    speed = np.linalg.norm(va, axis=1)
    cd, cl = coefficients(RADIUS * om / speed, aero)
    k = aero["rho"] * AREA / (2 * MASS)
    acc = k * speed[:, None] * (cl[:, None] * np.cross(axis, va) - cd[:, None] * va)
    acc[:, 2] -= G
    return acc, -aero["decay"] * om * speed


def simulate(v0, spin_rpm, tilt=0.0, aero=DEFAULT_AERO, wind=None, cp_d=None,
             dt=0.01, t_max=12.0, keep_path=False, stop="landing", land_h=0.0):
    """Fly every shot until it comes back down to height land_h and report checkpoint, apex and
    landing events. land_h = 0 is the level landing (tee height); -launch_z reaches the ground.

    v0: (N, 3) launch velocity (d, l, h); spin_rpm, tilt, land_h: (N,) or scalar;
    wind: (N, 3) or (3,); cp_d: (N, K) downrange distances at which to report crossings.
    stop="cps" ends the integration once every shot has crossed every checkpoint.
    """
    vel = np.array(v0, dtype=float)
    n = len(vel)
    pos = np.zeros((n, 3))
    om = np.broadcast_to(np.asarray(spin_rpm, float), (n,)) * RPM_TO_RAD
    axis = spin_axis(vel, np.broadcast_to(np.asarray(tilt, float), (n,)))
    wind = np.zeros((n, 3)) if wind is None else np.broadcast_to(np.asarray(wind, float), (n, 3))
    cp_d = np.empty((n, 0)) if cp_d is None else np.asarray(cp_d, float)
    land_h = np.broadcast_to(np.asarray(land_h, float), (n,))

    out = {k: np.full(n, np.nan) for k in
           ["apex_t", "apex_d", "apex_l", "apex_h", "landing_t", "landing_d", "landing_l"]}
    out["landing_vel"] = np.full((n, 3), np.nan)
    out["landing_spin"] = np.full(n, np.nan)
    out["axis"] = axis
    for k in ["cp_t", "cp_l", "cp_h"]:
        out[k] = np.full(cp_d.shape, np.nan)
    path, times = [pos.copy()], [0.0]

    t = 0.0
    while t < t_max and np.isnan(out["cp_t"] if stop == "cps" else out["landing_t"]).any():
        a1, w1 = _accel(vel, om, axis, wind, aero)
        v2, o2 = vel + 0.5 * dt * a1, om + 0.5 * dt * w1
        a2, w2 = _accel(v2, o2, axis, wind, aero)
        v3, o3 = vel + 0.5 * dt * a2, om + 0.5 * dt * w2
        a3, w3 = _accel(v3, o3, axis, wind, aero)
        v4, o4 = vel + dt * a3, om + dt * w3
        a4, w4 = _accel(v4, o4, axis, wind, aero)
        new_pos = pos + dt / 6 * (vel + 2 * v2 + 2 * v3 + v4)
        new_vel = vel + dt / 6 * (a1 + 2 * a2 + 2 * a3 + a4)
        new_om = om + dt / 6 * (w1 + 2 * w2 + 2 * w3 + w4)

        for j in range(cp_d.shape[1]):
            m = np.isnan(out["cp_t"][:, j]) & (pos[:, 0] < cp_d[:, j]) & (new_pos[:, 0] >= cp_d[:, j])
            if m.any():
                f = (cp_d[m, j] - pos[m, 0]) / (new_pos[m, 0] - pos[m, 0])
                out["cp_t"][m, j] = t + f * dt
                out["cp_l"][m, j] = pos[m, 1] + f * (new_pos[m, 1] - pos[m, 1])
                out["cp_h"][m, j] = pos[m, 2] + f * (new_pos[m, 2] - pos[m, 2])

        m = np.isnan(out["apex_t"]) & (vel[:, 2] > 0) & (new_vel[:, 2] <= 0)
        if m.any():
            f = vel[m, 2] / (vel[m, 2] - new_vel[m, 2])
            out["apex_t"][m] = t + f * dt
            for c, name in enumerate("dlh"):
                out[f"apex_{name}"][m] = pos[m, c] + f * (new_pos[m, c] - pos[m, c])

        m = (np.isnan(out["landing_t"]) & (pos[:, 2] > land_h) & (new_pos[:, 2] <= land_h)
             & (new_vel[:, 2] < 0))
        if m.any():
            f = (pos[m, 2] - land_h[m]) / (pos[m, 2] - new_pos[m, 2])
            out["landing_t"][m] = t + f * dt
            out["landing_d"][m] = pos[m, 0] + f * (new_pos[m, 0] - pos[m, 0])
            out["landing_l"][m] = pos[m, 1] + f * (new_pos[m, 1] - pos[m, 1])
            out["landing_vel"][m] = vel[m] + f[:, None] * (new_vel[m] - vel[m])
            out["landing_spin"][m] = (om[m] + f * (new_om[m] - om[m])) / RPM_TO_RAD

        pos, vel, om, t = new_pos, new_vel, new_om, t + dt
        if keep_path:
            path.append(pos.copy())
            times.append(t)

    if keep_path:
        out["path"], out["path_t"] = np.stack(path), np.array(times)
    return out


### `src/flight/calibration.py`
Joint sparse least-squares calibration of aerodynamics, per-shot nuisances and block wind.

In [ ]:
%%writefile src/flight/calibration.py
"""Calibrate the flight model on training shots, whose launch spin is known.

Global aerodynamic coefficients are fitted jointly with per-shot nuisance parameters (spin-axis tilt
and a small start offset, since the launch position in the data is the bay's nominal spot) and a
per-block wind, against every measured point: the four checkpoints plus apex and level landing.

    python -m flight.calibration [max_nfev]    writes outputs/calibration.json and calibration_shots.csv
"""
import json
import sys

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix

from data.dataset import POINTS, ROOT, load
from flight.simulator import DEFAULT_AERO, simulate

GLOBAL0 = {"cd0": 0.20, "cd1": 0.25, "cl_max": 0.45, "s_half": 0.12, "decay": 1.0}  # decay in 1e-3/m
GLOBAL_BOUNDS = {"cd0": (0.05, 0.5), "cd1": (0.0, 1.5), "cl_max": (0.05, 1.5), "s_half": (0.01, 1.0),
                 "decay": (0.0, 10.0)}
SHOT = {"tilt": (0.5, 1.2), "dd": (1.0, 3.0), "dl": (0.5, 2.0), "dh": (0.3, 1.0)}  # (prior sd, bound)
WIND_SD, WIND_BOUND = 3.0, 15.0
SIGMA = {"cp_t": 0.01, "cp_l": 0.10, "cp_h": 0.10, "apex_t": 0.08, "apex_d": 2.0, "apex_l": 0.8,
         "apex_h": 0.4, "landing_t": 0.05, "landing_d": 1.0, "landing_l": 1.0}
EVENTS = ["apex_t", "apex_d", "apex_l", "apex_h", "landing_t", "landing_d", "landing_l"]
ROWS = 12 + len(EVENTS)


def aero_from(values):
    aero = dict(DEFAULT_AERO)
    aero.update(dict(zip(GLOBAL0, values)))
    aero["decay"] *= 1e-3
    return aero


class Calibration:
    def __init__(self, df):
        self.n = len(df)
        self.v0 = df[["vd", "vl", "vh"]].to_numpy()
        self.spin = df.launch_spin_rate.to_numpy()
        self.cp_d = df[[f"{p}_d" for p in POINTS]].to_numpy()
        self.cp = {c: df[[f"{p}_{c}" for p in POINTS]].to_numpy() for c in "tlh"}
        self.ev = {k: df[k].to_numpy() for k in EVENTS}
        self.blocks, self.bidx = np.unique(df.block.to_numpy(), return_inverse=True)
        self.nb = len(self.blocks)
        self.ng = len(GLOBAL0)

    def unpack(self, x):
        g = x[:self.ng]
        tilt, dd, dl, dh = x[self.ng:self.ng + 4 * self.n].reshape(4, self.n)
        wind = x[self.ng + 4 * self.n:].reshape(2, self.nb)
        return g, tilt, dd, dl, dh, wind

    def wind_vectors(self, wind):
        return np.stack([wind[0, self.bidx], wind[1, self.bidx], np.zeros(self.n)], axis=1)

    def run(self, x):
        g, tilt, dd, dl, dh, wind = self.unpack(x)
        o = simulate(self.v0, self.spin, tilt, aero_from(g), wind=self.wind_vectors(wind),
                     cp_d=self.cp_d - dd[:, None])
        shift = {"d": dd, "l": dl, "h": dh, "t": 0.0}
        pred = {"cp_t": o["cp_t"], "cp_l": o["cp_l"] + dl[:, None], "cp_h": o["cp_h"] + dh[:, None]}
        for k in EVENTS:
            pred[k] = o[k] + shift[k[-1]]
        return pred, o

    def residuals(self, x):
        pred, _ = self.run(x)
        shot = [(pred[f"cp_{c}"] - self.cp[c]) / SIGMA[f"cp_{c}"] for c in "tlh"]
        shot += [((pred[k] - self.ev[k]) / SIGMA[k])[:, None] for k in EVENTS]
        shot = np.nan_to_num(np.hstack(shot), nan=50.0).ravel()
        _, tilt, dd, dl, dh, wind = self.unpack(x)
        prior = [p / SHOT[name][0] for name, p in zip(SHOT, (tilt, dd, dl, dh))] + [wind.ravel() / WIND_SD]
        return np.concatenate([shot] + prior)

    def sparsity(self):
        nshot = self.n * ROWS
        ncol = self.ng + 4 * self.n + 2 * self.nb
        S = lil_matrix((nshot + ncol - self.ng, ncol), dtype=np.int8)
        for i in range(self.n):
            rows = slice(i * ROWS, (i + 1) * ROWS)
            S[rows, :self.ng] = 1
            for p in range(4):
                S[rows, self.ng + p * self.n + i] = 1
            for c in range(2):
                S[rows, self.ng + 4 * self.n + c * self.nb + self.bidx[i]] = 1
        for k in range(ncol - self.ng):
            S[nshot + k, self.ng + k] = 1
        return S

    def fit(self, max_nfev=40, verbose=2):
        x0 = np.concatenate([list(GLOBAL0.values()), np.zeros(4 * self.n + 2 * self.nb)])
        lo = [b[0] for b in GLOBAL_BOUNDS.values()] + sum([[-SHOT[k][1]] * self.n for k in SHOT], [])
        hi = [b[1] for b in GLOBAL_BOUNDS.values()] + sum([[SHOT[k][1]] * self.n for k in SHOT], [])
        lo += [-WIND_BOUND] * 2 * self.nb
        hi += [WIND_BOUND] * 2 * self.nb
        return least_squares(self.residuals, x0, jac_sparsity=self.sparsity(), bounds=(lo, hi),
                             x_scale="jac", diff_step=1e-4, max_nfev=max_nfev, verbose=verbose)


def report(cal, x, df):
    pred, o = cal.run(x)
    g, tilt, dd, dl, dh, wind = cal.unpack(x)
    print("\nglobal:", {k: round(v, 4) for k, v in zip(GLOBAL0, g)})
    for c in "tlh":
        print(f"cp_{c} rms per checkpoint:", np.sqrt(np.nanmean((pred[f'cp_{c}'] - cal.cp[c]) ** 2, 0)).round(3))
    for k in EVENTS:
        r = pred[k] - cal.ev[k]
        print(f"{k:10s} bias {np.nanmean(r):7.3f}  rms {np.sqrt(np.nanmean(r ** 2)):6.3f}")
    land = np.hypot(pred["landing_d"] - cal.ev["landing_d"], pred["landing_l"] - cal.ev["landing_l"])
    print("landing euclid mean", land.mean().round(3))
    for name, v in zip(SHOT, (tilt, dd, dl, dh)):
        print(f"{name:5s} mean {v.mean():7.3f} sd {v.std():6.3f}")
    print("wind (d, l) per block:\n", pd.DataFrame(wind.T, index=cal.blocks, columns=["wd", "wl"]).round(2).T)
    resid = pd.DataFrame({"landing_d": pred["landing_d"] - cal.ev["landing_d"],
                          "apex_h": pred["apex_h"] - cal.ev["apex_h"], "landing_t": pred["landing_t"] - cal.ev["landing_t"]})
    for col, bins in [("launch_spin_rate", 4), ("speed", 4), ("vla", 4)]:
        print(resid.groupby(pd.qcut(df[col], bins), observed=True).mean().round(3))


if __name__ == "__main__":
    train, _ = load()
    cal = Calibration(train)
    result = cal.fit(max_nfev=int(sys.argv[1]) if len(sys.argv) > 1 else 40)
    report(cal, result.x, train)
    g, tilt, dd, dl, dh, wind = cal.unpack(result.x)
    out = {"aero": aero_from(g), "wind": {int(b): [float(w[0]), float(w[1])] for b, w in zip(cal.blocks, wind.T)}}
    (ROOT / "outputs" / "calibration.json").write_text(json.dumps(out, indent=2))
    pd.DataFrame({"track_id": train.track_id, "tilt": tilt, "dd": dd, "dl": dl, "dh": dh}).to_csv(
        ROOT / "outputs" / "calibration_shots.csv", index=False)


### `src/flight/inversion.py`
Batched Levenberg-Marquardt, checkpoint inversion and trajectory reconciliation.

In [ ]:
%%writefile src/flight/inversion.py
"""Per-shot inference from the checkpoints, and reconciling flights with a predicted apex and landing.

`invert` is the netted-range problem: a MAP fit of spin, spin-axis tilt and start offsets to the four
checkpoint crossings under the calibrated flight model. `reconcile` fits per-shot drag and lift
multipliers (plus tilt and offsets) so the simulated flight passes through the checkpoints and a given
apex and landing, turning point predictions into physically consistent full trajectories. Both are
solved with `batched_lm`, which runs hundreds of small least-squares problems as one.
"""
import json

import numpy as np
import pandas as pd

from data.dataset import POINTS, ROOT
from flight.simulator import simulate

PARAMS = ["spin", "tilt", "dd", "dl", "dh"]  # spin in thousands of rpm
RECONCILE_PARAMS = ["log_cd", "log_cl", "tilt", "dd", "dl", "dh"]
RECONCILE_PRIOR_SD = np.array([0.5, 0.5, 0.5, 1.0, 0.5, 0.5])
EVENTS = ["apex_t", "apex_d", "apex_l", "apex_h", "landing_t", "landing_d", "landing_l"]


def batched_lm(resid, theta0, step, lower, upper, n_iter=12):
    """Levenberg-Marquardt for many independent problems: minimise sum(resid(theta, idx)**2) per row.

    resid(theta, idx) evaluates problems `idx` at parameters theta (len(idx), P) and returns
    (len(idx), M) residuals; all forward-difference Jacobians are taken in one batched call.
    """
    n, p = theta0.shape
    rows = np.arange(n)
    theta = np.clip(np.array(theta0, dtype=float), lower, upper)
    r = resid(theta, rows)
    cost = (r**2).sum(1)
    lam = np.full(n, 1e-2)
    for _ in range(n_iter):
        pert = (theta[:, None, :] + np.eye(p)[None] * step).reshape(-1, p)
        rp = resid(pert, np.repeat(rows, p)).reshape(n, p, -1)
        J = ((rp - r[:, None, :]) / step[None, :, None]).transpose(0, 2, 1)  # (N, M, P)
        JtJ = J.transpose(0, 2, 1) @ J
        grad = J.transpose(0, 2, 1) @ r[..., None]
        damp = lam[:, None, None] * np.eye(p)[None] * np.diagonal(JtJ, axis1=1, axis2=2)[:, None, :]
        trial = np.clip(theta - np.linalg.solve(JtJ + damp + 1e-9 * np.eye(p), grad)[..., 0], lower, upper)
        r_new = resid(trial, rows)
        cost_new = (r_new**2).sum(1)
        ok = cost_new < cost
        theta[ok], r[ok], cost[ok] = trial[ok], r_new[ok], cost_new[ok]
        lam = np.where(ok, lam * 0.3, lam * 10)
    return theta, cost


def load_calibration():
    cal = json.loads((ROOT / "outputs" / "calibration.json").read_text())
    cal["wind"] = {int(k): v for k, v in cal["wind"].items()}
    return cal


def wind_for(df, wind_by_block):
    """Per-shot wind from its hitting block; unseen blocks get the mean, which carries the calibration's
    constant offset (a zero default would bias the physics for a brand-new session)."""
    default = np.mean(list(wind_by_block.values()), axis=0) if wind_by_block else [0.0, 0.0]
    w = np.array([wind_by_block.get(int(b), default) for b in df.block])
    return np.column_stack([w, np.zeros(len(df))])


def arrays(df):
    return dict(v0=df[["vd", "vl", "vh"]].to_numpy(),
                cp_d=df[[f"{p}_d" for p in POINTS]].to_numpy(),
                obs=np.concatenate([df[[f"{p}_{c}" for p in POINTS]].to_numpy() for c in "tlh"], axis=1))


def scaled_aero(aero, log_cd=0.0, log_cl=0.0):
    return dict(aero, cd0=aero["cd0"] * np.exp(log_cd), cd1=aero["cd1"] * np.exp(log_cd),
                cl_max=aero["cl_max"] * np.exp(log_cl))


def invert(df, aero, wind, prior_mean, prior_sd, sigma_cp=(0.01, 0.1, 0.1), n_iter=12):
    """MAP estimate of PARAMS per shot. prior_mean, prior_sd: (N, 5) in PARAMS order."""
    a = arrays(df)
    obs_sd = np.repeat(sigma_cp, 4)

    def resid(theta, idx):
        spin, tilt, dd, dl, dh = theta.T
        o = simulate(a["v0"][idx], spin * 1000, tilt, aero, wind=wind[idx], cp_d=a["cp_d"][idx] - dd[:, None],
                     stop="cps", t_max=5.0)
        pred = np.concatenate([o["cp_t"], o["cp_l"] + dl[:, None], o["cp_h"] + dh[:, None]], axis=1)
        r = np.concatenate([(pred - a["obs"][idx]) / obs_sd, (theta - prior_mean[idx]) / prior_sd[idx]], axis=1)
        return np.nan_to_num(r, nan=50.0)

    theta, cost = batched_lm(resid, prior_mean, step=np.array([0.05, 0.005, 0.01, 0.01, 0.01]),
                             lower=np.array([0.3, -1.2, -3.0, -2.0, -1.0]),
                             upper=np.array([15.0, 1.2, 3.0, 2.0, 1.0]), n_iter=n_iter)
    return pd.DataFrame(theta, columns=PARAMS, index=df.index), cost


def fly(df, theta, aero, wind, keep_path=False):
    """Full flight from the fitted parameters; returns local-frame predictions (and paths if asked)."""
    spin, tilt, dd, dl, dh = theta[PARAMS].to_numpy().T
    o = simulate(df[["vd", "vl", "vh"]].to_numpy(), spin * 1000, tilt, aero, wind=wind, keep_path=keep_path)
    pred = pd.DataFrame(index=df.index)
    pred["launch_spin_rate"] = spin * 1000
    pred["apex_t"] = o["apex_t"]
    pred["apex_d"], pred["apex_l"], pred["apex_h"] = o["apex_d"] + dd, o["apex_l"] + dl, o["apex_h"] + dh
    pred["landing_t"] = o["landing_t"]
    pred["landing_d"], pred["landing_l"], pred["landing_h"] = o["landing_d"] + dd, o["landing_l"] + dl, 0.0
    return (pred, o) if keep_path else pred


def reconcile(df, targets, spin_rpm, aero, wind, prior_sd=RECONCILE_PRIOR_SD, n_iter=15):
    """Fit a flight through the checkpoints and the apex/landing in `targets` (EVENTS columns)."""
    a = arrays(df)
    obs = np.column_stack([a["obs"]] + [targets[k].to_numpy() for k in EVENTS])
    obs_sd = np.array([0.015] * 4 + [0.15] * 4 + [0.3] * 4 + [0.08, 2.0, 0.8, 0.4, 0.05, 1.0, 1.0])

    def resid(theta, idx):
        log_cd, log_cl, tilt, dd, dl, dh = theta.T
        o = simulate(a["v0"][idx], spin_rpm[idx], tilt, scaled_aero(aero, log_cd, log_cl), wind=wind[idx],
                     cp_d=a["cp_d"][idx] - dd[:, None])
        pred = np.column_stack([o["cp_t"], o["cp_l"] + dl[:, None], o["cp_h"] + dh[:, None],
                                o["apex_t"], o["apex_d"] + dd, o["apex_l"] + dl, o["apex_h"] + dh,
                                o["landing_t"], o["landing_d"] + dd, o["landing_l"] + dl])
        return np.nan_to_num(np.concatenate([(pred - obs[idx]) / obs_sd, theta / prior_sd], axis=1), nan=50.0)

    theta, _ = batched_lm(resid, np.zeros((len(df), 6)), step=np.array([0.01, 0.01, 0.005, 0.01, 0.01, 0.01]),
                          lower=np.array([-1.5, -1.5, -1.2, -3.0, -2.0, -1.0]),
                          upper=np.array([1.5, 1.5, 1.2, 3.0, 2.0, 1.0]), n_iter=n_iter)
    return pd.DataFrame(theta, columns=RECONCILE_PARAMS, index=df.index)


### `src/flight/ground.py`
Bounce and roll: Penner (2002) crater impact, friction, rolling resistance and grass drag.

In [ ]:
%%writefile src/flight/ground.py
"""Bounce and roll after the ball reaches the ground.

Impacts follow the crater model of A. R. Penner, "The run of a golf ball", Can. J. Phys. 80 (2002):
the turf gives way and tilts the effective contact normal back against the direction of travel by an
angle that grows with impact speed and steepness; the normal rebound uses a speed-dependent
coefficient of restitution and the tangential impulse is capped by Coulomb friction, or ends in
rolling. Between impacts the ball hops ballistically with drag; once the rebound is negligible it rolls
to rest under rolling resistance plus a speed-proportional grass drag. Turf constants are generic
range-grass values, not measured at this range.
"""
import numpy as np

from flight.simulator import AREA, G, MASS, RADIUS

TURF = dict(mu=0.43, crater_deg=15.4, crater_speed=18.6, crater_angle=44.4,
            roll_resistance=0.3, grass_drag=0.8, min_rebound=1.2, hop_cd=0.25, rho=1.19)


def restitution(vn):
    """Normal coefficient of restitution against normal impact speed (m/s)."""
    return np.where(vn <= 20.0, 0.510 - 0.0375 * vn + 0.000903 * vn**2, 0.120)


def impact(vel, omega, turf=TURF):
    """One turf impact. vel (3,) m/s and omega (3,) rad/s in a z-up frame; returns both after impact."""
    horiz = np.hypot(vel[0], vel[1])
    angle = np.degrees(np.arctan2(-vel[2], horiz))
    crater = np.radians(turf["crater_deg"] * (np.linalg.norm(vel) / turf["crater_speed"])
                        * (angle / turf["crater_angle"]))
    heading = np.array([vel[0] / horiz, vel[1] / horiz, 0.0]) if horiz > 1e-9 else np.array([1.0, 0, 0])
    n = np.cos(crater) * np.array([0.0, 0.0, 1.0]) - np.sin(crater) * heading
    vn = vel @ n
    if vn >= 0:
        return vel, omega
    jn = (1 + restitution(-vn)) * -vn           # normal impulse per unit mass
    r_c = -RADIUS * n                            # contact point relative to the centre
    contact = vel + np.cross(omega, r_c)
    slip = contact - (contact @ n) * n
    s = np.linalg.norm(slip)
    jt = -min(turf["mu"] * jn, 2 / 7 * s) * slip / s if s > 1e-9 else np.zeros(3)
    return vel + jn * n + jt, omega + np.cross(r_c, jt) / (0.4 * RADIUS**2)


def bounce_and_roll(pos, vel, omega, turf=TURF, dt=0.005, max_impacts=12):
    """Follow the ball from its first ground contact at pos (ground height = pos[2]) until it stops.

    Returns times, path (T, 3), the impact points, and the rest position.
    """
    pos, vel, omega = np.array(pos, float), np.array(vel, float), np.array(omega, float)
    ground = pos[2]
    k = turf["rho"] * AREA / (2 * MASS) * turf["hop_cd"]
    t, times, path, impacts = 0.0, [0.0], [pos.copy()], [pos.copy()]
    for _ in range(max_impacts):
        vel, omega = impact(vel, omega, turf)
        if vel[2] < turf["min_rebound"]:
            break
        while True:  # a hop: gravity and drag, semi-implicit Euler
            vel = vel + dt * (-k * np.linalg.norm(vel) * vel - np.array([0.0, 0.0, G]))
            pos = pos + dt * vel
            t += dt
            landed = pos[2] <= ground and vel[2] < 0
            if landed:
                pos[2] = ground
            times.append(t)
            path.append(pos.copy())
            if landed:
                impacts.append(pos.copy())
                break

    # Roll: dv/dt = -(a + beta v), solved in closed form until the ball stops.
    roll = np.array([vel[0], vel[1], 0.0])
    speed = np.linalg.norm(roll)
    a, beta = turf["roll_resistance"] * G, turf["grass_drag"]
    if speed > 1e-6:
        direction = roll / speed
        stop = np.log1p(beta * speed / a) / beta
        for tau in np.linspace(0.0, stop, 60)[1:]:
            dist = (speed + a / beta) * (1 - np.exp(-beta * tau)) / beta - a / beta * tau
            times.append(t + tau)
            path.append(pos + direction * dist)
    return dict(t=np.array(times), path=np.array(path), impacts=np.array(impacts), rest=path[-1])


### `src/modelling/evaluation.py`
Error components in competition units and a proxy for the hidden composite score.

In [ ]:
%%writefile src/modelling/evaluation.py
"""Error components in competition units, and a proxy for the hidden composite score."""
import numpy as np
import pandas as pd

from data.dataset import LOCAL_TARGETS

# Proxy for the hidden composite metric: landing heaviest, then apex, then times, spin least.
WEIGHTS = {"landing": 0.40, "apex": 0.25, "apex_t": 0.125, "landing_t": 0.125, "spin": 0.10}


def errors(truth, pred):
    """Per-shot error components in competition units; distances are rotation-invariant."""
    e = pd.DataFrame(index=truth.index)
    for p in ["landing", "apex"]:
        ph = pred[f"{p}_h"].to_numpy() if f"{p}_h" in pred else 0.0
        e[p] = np.sqrt((truth[f"{p}_d"].to_numpy() - pred[f"{p}_d"].to_numpy()) ** 2
                       + (truth[f"{p}_l"].to_numpy() - pred[f"{p}_l"].to_numpy()) ** 2
                       + (truth[f"{p}_h"].to_numpy() - ph) ** 2)
    e["apex_t"] = np.abs(truth.apex_t.to_numpy() - pred.apex_t.to_numpy())
    e["landing_t"] = np.abs(truth.landing_t.to_numpy() - pred.landing_t.to_numpy())
    e["spin"] = np.abs(truth.launch_spin_rate.to_numpy() - pred.launch_spin_rate.to_numpy())
    return e


def mean_scales(train):
    """Error of the train-mean predictor for each component, used to normalise the composite."""
    mean = pd.DataFrame({c: np.full(len(train), train[c].mean()) for c in LOCAL_TARGETS})
    mean["landing_h"] = 0.0
    return errors(train, mean).mean()


def summarise(err, scales):
    row = err.mean()
    row["composite"] = sum(w * row[k] / scales[k] for k, w in WEIGHTS.items())
    return row


### `src/modelling/pipeline.py`
Final hybrid model with honest cross-validation, ablation switches and submission.

In [ ]:
%%writefile src/modelling/pipeline.py
"""Final model: calibrated flight physics plus a Gaussian-process correction layer.

Everything learned from labels (aerodynamic calibration, block wind, spin model, GP layer) is refitted
inside each cross-validation fold, so the CV score is an honest estimate for the test set.

    python -m modelling.pipeline --cv --submit [--layer gp_feature] [--no-wind] [--groups session]

Ablations (CV only): --oracle-spin feeds the true launch-monitor spin instead of the LightGBM estimate;
--no-fingerprints drops the kinematic features from the spin model and the GP layer.
Run from src/ or with PYTHONPATH=src.
"""
import argparse
import json
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from data.dataset import LOCAL_TARGETS, ROOT, load, to_submission
from data.features import BASE_FEATURES, feature_frame
from flight.calibration import Calibration, aero_from
from flight.inversion import fly, invert, reconcile, wind_for
from modelling.evaluation import errors, mean_scales, summarise

warnings.filterwarnings("ignore")
RAW_GP_FEATURES = ["vd", "vl", "vh", "elevated", "cp2_l", "cp2_h", "cp4_t", "cp4_l", "cp4_h"]
GP_FEATURES = RAW_GP_FEATURES + ["lift_proxy", "time_excess", "side_excess"]
POSITION_TARGETS = LOCAL_TARGETS[1:]


def lgbm():
    return lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.7, n_jobs=1, verbose=-1)


def gp_fit_predict(x_fit, y_fit, x_pred):
    scaler = StandardScaler().fit(x_fit)
    kernel = ConstantKernel() * RBF(np.ones(x_fit.shape[1]), (1e-2, 1e3)) + WhiteKernel(1e-2)
    gp = GaussianProcessRegressor(kernel, normalize_y=True).fit(scaler.transform(x_fit), y_fit)
    return gp.predict(scaler.transform(x_pred))


def calibrate(df, max_nfev=40):
    cal = Calibration(df)
    g, _, dd, dl, dh, wind = cal.unpack(cal.fit(max_nfev=max_nfev, verbose=0).x)
    return dict(aero=aero_from(g), wind={int(b): [float(w[0]), float(w[1])] for b, w in zip(cal.blocks, wind.T)},
                nuisance={k: [float(v.mean()), float(v.std())] for k, v in (("dd", dd), ("dl", dl), ("dh", dh))})


def physics_predictions(df, spin_rpm, cal, use_wind):
    """Invert the checkpoints with spin fixed at the given estimate, then fly each shot to landing."""
    n = len(df)
    wind = wind_for(df, cal["wind"]) if use_wind else np.zeros((n, 3))
    nuis = cal["nuisance"]
    prior_mean = np.column_stack([spin_rpm / 1000, np.zeros(n)] + [np.full(n, nuis[k][0]) for k in ("dd", "dl", "dh")])
    prior_sd = np.tile([0.01, 0.3] + [nuis[k][1] for k in ("dd", "dl", "dh")], (n, 1))
    theta, _ = invert(df, cal["aero"], wind, prior_mean, prior_sd, sigma_cp=(0.015, 0.15, 0.3))
    return fly(df, theta, cal["aero"], wind), wind


def fit_predict(fit_df, pred_df, layer="gp_feature", use_wind=True, oracle_spin=False, fingerprints=True):
    started = time.time()
    gp_features = GP_FEATURES if fingerprints else RAW_GP_FEATURES
    with threadpool_limits(1):
        cal = calibrate(fit_df)
        if fingerprints:
            x_fit, x_pred = feature_frame(fit_df), feature_frame(pred_df)
        else:
            x_fit, x_pred = fit_df[BASE_FEATURES], pred_df[BASE_FEATURES]
        if oracle_spin:  # ablation: as if the launch spin were measured
            spin_fit, spin_pred = fit_df.launch_spin_rate.to_numpy(), pred_df.launch_spin_rate.to_numpy()
        else:
            spin_fit = np.zeros(len(fit_df))
            for a, b in KFold(5, shuffle=True, random_state=0).split(x_fit):
                spin_fit[b] = lgbm().fit(x_fit.iloc[a], fit_df.launch_spin_rate.iloc[a]).predict(x_fit.iloc[b])
            spin_pred = lgbm().fit(x_fit, fit_df.launch_spin_rate).predict(x_pred)
        phys_fit, _ = physics_predictions(fit_df, spin_fit, cal, use_wind)
        phys_pred, wind_pred = physics_predictions(pred_df, spin_pred, cal, use_wind)

        pred = pd.DataFrame({"launch_spin_rate": spin_pred}, index=pred_df.index)
        for target in POSITION_TARGETS:
            y, pf, pp = fit_df[target], phys_fit[target], phys_pred[target]
            if layer == "gp_residual":
                pred[target] = pp.to_numpy() + gp_fit_predict(x_fit[gp_features], y - pf, x_pred[gp_features])
            elif layer == "gp_feature":
                pred[target] = gp_fit_predict(x_fit[gp_features].assign(phys=pf), y,
                                              x_pred[gp_features].assign(phys=pp))
            else:  # physics only
                pred[target] = pp
        pred["landing_h"] = 0.0
    print(f"fold of {len(fit_df)} → {len(pred_df)} shots done in {time.time() - started:.0f}s", flush=True)
    return pred, dict(cal=cal, physics=phys_pred, wind=wind_pred)


def cross_validate(train, layer, use_wind, k=5, groups=None, oracle_spin=False, fingerprints=True):
    splitter = GroupKFold(k) if groups is not None else KFold(k, shuffle=True, random_state=42)
    folds = list(splitter.split(train, groups=groups))
    parts = Parallel(n_jobs=k)(delayed(fit_predict)(train.iloc[a], train.iloc[b], layer, use_wind, oracle_spin,
                                                    fingerprints) for a, b in folds)
    return pd.concat([p for p, _ in parts]).loc[train.index], pd.concat([e["physics"] for _, e in parts]).loc[train.index]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--cv", action="store_true")
    parser.add_argument("--submit", action="store_true")
    parser.add_argument("--layer", default="gp_feature", choices=["gp_feature", "gp_residual", "physics"])
    parser.add_argument("--no-wind", action="store_true")
    parser.add_argument("--groups", default="none", choices=["none", "session"])
    parser.add_argument("--oracle-spin", action="store_true")
    parser.add_argument("--no-fingerprints", action="store_true")
    args = parser.parse_args()
    if args.submit and args.oracle_spin:
        parser.error("--oracle-spin needs the true spin, which the test set does not have")
    use_wind, fingerprints = not args.no_wind, not args.no_fingerprints
    tag = (f"{args.layer}{'' if use_wind else '_nowind'}{'_oraclespin' if args.oracle_spin else ''}"
           f"{'' if fingerprints else '_nofingerprints'}{'_bysession' if args.groups == 'session' else ''}")

    train, test = load()
    scales = mean_scales(train)
    if args.cv:
        groups = train.session if args.groups == "session" else None
        oof, phys = cross_validate(train, args.layer, use_wind, groups=groups, oracle_spin=args.oracle_spin,
                                   fingerprints=fingerprints)
        report = pd.DataFrame({"model": summarise(errors(train, oof), scales),
                               "physics_only": summarise(errors(train, phys.assign(launch_spin_rate=oof.launch_spin_rate)), scales)})
        print(report.round(3).T)
        oof.assign(track_id=train.track_id.values).to_csv(ROOT / "outputs" / f"pipeline_oof_{tag}.csv", index=False)
    if args.submit:
        pred, extra = fit_predict(train, test, args.layer, use_wind, fingerprints=fingerprints)
        (ROOT / "submissions").mkdir(exist_ok=True)
        to_submission(test, pred).to_csv(ROOT / "submissions" / "submission.csv", index=False)
        (ROOT / "outputs" / "calibration_final.json").write_text(json.dumps(extra["cal"], indent=2))
        params = reconcile(test, pred, pred.launch_spin_rate.to_numpy(), extra["cal"]["aero"], extra["wind"])
        params.assign(track_id=test.track_id.values).to_csv(ROOT / "outputs" / "test_trajectory_params.csv", index=False)
        pred.assign(track_id=test.track_id.values).to_csv(ROOT / "outputs" / "test_predictions_local.csv", index=False)
        print("submission written:", len(pred), "rows")


### `src/experiments/ml_studies.py`
Machine-learning studies and the what/why/when error analysis.

In [ ]:
%%writefile src/experiments/ml_studies.py
"""Machine-learning studies behind the final model choice, and the analysis of its errors.

    python -m experiments.ml_studies explore         kinematic spin signal, block effects, same-club runs
    python -m experiments.ml_studies baselines       ridge, extra trees, LightGBM and GP; random and leave-session-out CV
    python -m experiments.ml_studies neighbours      does the neighbouring training shot in time help?
    python -m experiments.ml_studies hybrid          physics prediction as GP feature vs residual base vs LightGBM feature
    python -m experiments.ml_studies error_analysis  what, why and when of the final model's out-of-fold errors

`hybrid` reads outputs/calibration.json from `python -m flight.calibration`; `error_analysis` also needs
outputs/aero_diag.csv and the pipeline's out-of-fold predictions. Run from src/ or with PYTHONPATH=src.
"""
import argparse
import json
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import kruskal, spearmanr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from threadpoolctl import threadpool_limits

from data.dataset import LOCAL_TARGETS, ROOT, load
from data.features import BASE_FEATURES, feature_frame, kinematic_features
from flight.inversion import fly, invert, load_calibration, wind_for
from modelling.evaluation import errors, mean_scales, summarise
from modelling.pipeline import GP_FEATURES, POSITION_TARGETS, gp_fit_predict, lgbm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
RANDOM_FOLDS = KFold(5, shuffle=True, random_state=42)
ERROR_FACTORS = {
    "spin_error": "spin misestimate",
    "aero_anomaly": "unusual drag or lift for this ball",
    "curvature": "curvature (spin-axis tilt)",
    "carry_beyond_net": "carry beyond the net",
    "vla": "launch angle",
    "novelty": "distance to similar training shots",
    "upper_deck": "upper-deck bay",
}


def explore(train):
    kin = kinematic_features(train)
    tr = pd.concat([train, kin], axis=1)
    print("blocks:", train.block.nunique(), "| training shots per block:", train.block.value_counts().sort_index().tolist())
    print("\ncorrelation with launch spin:")
    print(tr[list(kin.columns) + ["speed", "vla", "cp4_t", "cp4_h"]].corrwith(tr.launch_spin_rate).round(3).to_string())

    x = pd.concat([train[BASE_FEATURES], kin], axis=1)
    print("\ntarget            OOF MAE   corr(residual, leave-one-out block mean residual)")
    for target in ["landing_d", "landing_l", "apex_h", "apex_d", "landing_t", "launch_spin_rate"]:
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        resid = tr[target] - cross_val_predict(model, x, tr[target], cv=RANDOM_FOLDS)
        g = resid.groupby(tr.block)
        loo = (g.transform("sum") - resid) / (g.transform("count") - 1)
        ok = g.transform("count") > 1
        print(f"{target:16s} {resid.abs().mean():8.3f}   {np.corrcoef(resid[ok], loo[ok])[0, 1]:.3f}")

    s = tr.sort_values("launch_time")
    same = s.block.eq(s.block.shift())
    for col in ["launch_spin_rate", "speed", "vla", "landing_l"]:
        print(f"lag-1 corr {col:18s} {np.corrcoef(s[col][same], s[col].shift()[same])[0, 1]:.3f}")


def _oof(train, x, folds, fit_predict):
    pred = pd.DataFrame(index=train.index, columns=LOCAL_TARGETS, dtype=float)
    for fit, val in folds:
        for target in LOCAL_TARGETS:
            pred.iloc[val, pred.columns.get_loc(target)] = fit_predict(x.iloc[fit], train[target].iloc[fit], x.iloc[val])
    pred["landing_h"] = 0.0
    return pred


def _sklearn(make):
    return lambda x_fit, y_fit, x_pred: make().fit(x_fit, y_fit).predict(x_pred)


def baselines(train):
    x = feature_frame(train)
    scales = mean_scales(train)
    models = {
        "ridge_poly2": (_sklearn(lambda: make_pipeline(StandardScaler(), PolynomialFeatures(2), StandardScaler(),
                                                       RidgeCV(alphas=np.logspace(-1, 4, 16)))), None),
        "extratrees": (_sklearn(lambda: ExtraTreesRegressor(500, min_samples_leaf=2, max_features=0.5, n_jobs=-1,
                                                            random_state=0)), None),
        "lightgbm": (_sklearn(lgbm), None),
        "gp_ard": (gp_fit_predict, GP_FEATURES),
    }
    splits = {"kfold": list(RANDOM_FOLDS.split(x)), "session": list(GroupKFold(5).split(x, groups=train.session))}
    rows = {}
    for name, (fit_predict, cols) in models.items():
        for cv_name, folds in splits.items():
            rows[(name, cv_name)] = summarise(errors(train, _oof(train, x[cols] if cols else x, folds, fit_predict)), scales)
            print(name, cv_name, rows[(name, cv_name)].round(3).to_dict(), flush=True)
    print(pd.DataFrame(rows).T.round(3))


def _neighbour_features(query, pool):
    """Closest-in-time pool shots before and after each query shot within the same hitting block."""
    q = query[["launch_time", "block", "speed", "vla"]].reset_index().sort_values("launch_time")
    pl = pool[["launch_time", "block", "speed", "vla", "launch_spin_rate", "curve", "carry_ratio"]]
    pl = pl.assign(t_nb=pl.launch_time).sort_values("launch_time")
    out = pd.DataFrame(index=query.index)
    for side, direction in [("prev", "backward"), ("next", "forward")]:
        m = pd.merge_asof(q, pl, on="launch_time", by="block", direction=direction,
                          allow_exact_matches=False, suffixes=("", "_nb")).set_index("index")
        out[f"{side}_dt"] = (m.launch_time - m.t_nb).abs()
        out[f"{side}_spin"] = m.launch_spin_rate
        out[f"{side}_dspeed"] = m.speed - m.speed_nb
        out[f"{side}_dvla"] = m.vla - m.vla_nb
        out[f"{side}_curve"] = m.curve
        out[f"{side}_carry"] = m.carry_ratio
    return out


def neighbours(train):
    train = train.assign(curve=train.landing_l - train.cp4_l * train.landing_d / train.cp4_d,
                         carry_ratio=train.landing_d / train.cp4_t)
    x = feature_frame(train)
    scales = mean_scales(train)
    rows = {}
    for use in (False, True):
        pred = pd.DataFrame(index=train.index, columns=LOCAL_TARGETS, dtype=float)
        for fit, val in RANDOM_FOLDS.split(x):
            f, v = train.iloc[fit], train.iloc[val]
            xf, xv = x.iloc[fit], x.iloc[val]
            if use:
                xf = pd.concat([xf, _neighbour_features(f, f)], axis=1)
                xv = pd.concat([xv, _neighbour_features(v, f)], axis=1)
            for target in LOCAL_TARGETS:
                pred.loc[v.index, target] = lgbm().fit(xf, f[target]).predict(xv)
        pred["landing_h"] = 0.0
        rows["with neighbours" if use else "plain"] = summarise(errors(train, pred), scales)
    print(pd.DataFrame(rows).T.round(3))


def _hybrid_task(train, x, folds, physics, kind, variant, target, k):
    fit, val = folds[k]
    y = train[target]
    phys = physics[variant][target] if variant else None
    with threadpool_limits(1):
        if kind == "gp_plain":
            pred = gp_fit_predict(x[GP_FEATURES].iloc[fit], y.iloc[fit], x[GP_FEATURES].iloc[val])
        elif kind == "gp_feature":
            cols = x[GP_FEATURES].assign(phys=phys)
            pred = gp_fit_predict(cols.iloc[fit], y.iloc[fit], cols.iloc[val])
        elif kind == "gp_residual":
            pred = phys.iloc[val].to_numpy() + gp_fit_predict(x[GP_FEATURES].iloc[fit], (y - phys).iloc[fit],
                                                              x[GP_FEATURES].iloc[val])
        else:  # lgbm_feature
            cols = pd.concat([x, physics[variant][POSITION_TARGETS].add_prefix("phys_")], axis=1)
            pred = lgbm().fit(cols.iloc[fit], y.iloc[fit]).predict(cols.iloc[val])
    return kind, variant, target, val, pred


def hybrid(train):
    """Uses the all-train calibration, so wind is slightly optimistic; the pipeline's CV is the honest version."""
    x = feature_frame(train)
    scales = mean_scales(train)
    folds = list(RANDOM_FOLDS.split(x))
    cal = load_calibration()
    shots = pd.read_csv(ROOT / "outputs" / "calibration_shots.csv")
    spin_oof = np.zeros(len(train))
    for fit, val in folds:
        spin_oof[val] = lgbm().fit(x.iloc[fit], train.launch_spin_rate.iloc[fit]).predict(x.iloc[val])

    n = len(train)
    prior_mean = np.column_stack([spin_oof / 1000, np.zeros(n)] + [np.full(n, shots[k].mean()) for k in ("dd", "dl", "dh")])
    prior_sd = np.tile([0.01, 0.3] + [shots[k].std() for k in ("dd", "dl", "dh")], (n, 1))
    physics = {}
    for variant, wind in [("wind", wind_for(train, cal["wind"])), ("nowind", np.zeros((n, 3)))]:
        theta, _ = invert(train, cal["aero"], wind, prior_mean, prior_sd, sigma_cp=(0.015, 0.15, 0.3))
        physics[variant] = fly(train, theta, cal["aero"], wind)

    configs = [("gp_plain", None)] + [(k, v) for k in ["gp_feature", "gp_residual", "lgbm_feature"] for v in ["wind", "nowind"]]
    jobs = [delayed(_hybrid_task)(train, x, folds, physics, kind, v, t, k)
            for kind, v in configs for t in POSITION_TARGETS for k in range(5)]
    results = Parallel(n_jobs=20, verbose=5)(jobs)
    rows = {}
    for kind, v in configs:
        pred = pd.DataFrame(index=train.index, columns=POSITION_TARGETS, dtype=float)
        for kd, vr, target, val, p in results:
            if kd == kind and vr == v:
                pred.iloc[val, pred.columns.get_loc(target)] = p
        pred["launch_spin_rate"], pred["landing_h"] = spin_oof, 0.0
        rows[f"{kind}/{v}"] = summarise(errors(train, pred), scales)
    for v in ["wind", "nowind"]:
        rows[f"physics_only/{v}"] = summarise(errors(train, physics[v].assign(launch_spin_rate=spin_oof)), scales)
    print(pd.DataFrame(rows).T.round(3))


def _lag1(values, same):
    return np.corrcoef(values[same], np.roll(values, 1)[same])[0, 1]


def error_analysis(train, reps=2000, seed=0):
    """What, why and when of the final model's out-of-fold landing errors.

    Why: rank correlation of each factor with the relative landing error (error as % of carry), and the
    adjusted effect of each factor from a regression of log relative error on all standardised factors,
    both with bootstrap 95% intervals. When: Kruskal-Wallis across sessions, rank correlation with time of
    day and position in the hitting block, and the lag-1 autocorrelation of signed residuals between
    consecutive training shots against a within-block permutation null.
    Writes outputs/error_analysis.csv (per shot) and outputs/error_analysis.json (statistics).
    """
    rng = np.random.default_rng(seed)
    _, test = load()
    oof = pd.read_csv(ROOT / "outputs" / "pipeline_oof_gp_feature.csv").set_index("track_id").loc[train.track_id]
    diag = pd.read_csv(ROOT / "outputs" / "aero_diag.csv").set_index("track_id").loc[train.track_id]
    oof.index, diag.index = train.index, train.index
    err = errors(train, oof)

    t = pd.DataFrame({"track_id": train.track_id, "landing_error": err.landing,
                      "rel_error": 100 * err.landing / train.landing_d,
                      "res_d": oof.landing_d - train.landing_d, "res_l": oof.landing_l - train.landing_l,
                      "spin_error": (oof.launch_spin_rate - train.launch_spin_rate).abs(),
                      "aero_anomaly": np.hypot(diag.log_cd, diag.log_cl),
                      "curvature": np.degrees(diag.tilt.abs()),
                      "carry_beyond_net": train.landing_d - train.cp4_d,
                      "vla": train.vla, "upper_deck": train.elevated, "session": train.session})
    x = StandardScaler().fit_transform(feature_frame(train)[GP_FEATURES])
    t["novelty"] = NearestNeighbors(n_neighbors=11).fit(x).kneighbors(x)[0][:, 1:].mean(1)
    local = pd.to_datetime(train.launch_time, unit="s") + pd.Timedelta(hours=2)  # South African Standard Time
    t["local_hour"] = local.dt.hour + local.dt.minute / 60
    t["position_in_block"] = pd.concat([train, test]).groupby("block").launch_time.rank(pct=True).iloc[:len(train)].to_numpy()

    # Why
    factors = list(ERROR_FACTORS)
    y = np.log(t.rel_error.to_numpy())
    design = np.column_stack([np.ones(len(t)), StandardScaler().fit_transform(t[factors])])
    coef = np.linalg.lstsq(design, y, rcond=None)[0]
    r2 = 1 - ((y - design @ coef) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    boot_rho, boot_beta = [], []
    ranks = t[factors + ["rel_error"]].rank().to_numpy()
    for _ in range(reps):
        idx = rng.integers(0, len(t), len(t))
        boot_beta.append(np.linalg.lstsq(design[idx], y[idx], rcond=None)[0][1:])
        boot_rho.append([np.corrcoef(ranks[idx, i], ranks[idx, -1])[0, 1] for i in range(len(factors))])
    boot_rho, boot_beta = np.array(boot_rho), np.array(boot_beta)
    why = {f: dict(label=ERROR_FACTORS[f], rho=float(spearmanr(t[f], t.rel_error)[0]),
                   rho_ci=np.nanpercentile(boot_rho[:, i], [2.5, 97.5]).tolist(),
                   pct_per_sd=float(100 * np.expm1(coef[i + 1])),
                   pct_per_sd_ci=(100 * np.expm1(np.percentile(boot_beta[:, i], [2.5, 97.5]))).tolist())
           for i, f in enumerate(factors)}

    # What
    worst = t.landing_error >= t.landing_error.quantile(0.9)
    profile = {f: dict(worst=float(t.loc[worst, f].median()), rest=float(t.loc[~worst, f].median()))
               for f in factors + ["landing_error", "rel_error"]}
    profile["upper_deck"] = dict(worst=float(t.loc[worst, "upper_deck"].mean()), rest=float(t.loc[~worst, "upper_deck"].mean()))
    order = t.sort_values("landing_error")
    cases = {"best": order.track_id.iloc[0], "typical": order.track_id.iloc[len(order) // 2],
             "95th percentile": order.track_id.iloc[int(0.95 * len(order))], "worst": order.track_id.iloc[-1]}

    # When
    sessions = [g.to_numpy() for _, g in t.groupby("session").rel_error]
    s = t.assign(block=train.block, time=train.launch_time).sort_values("time")
    same = s.block.eq(s.block.shift()).to_numpy()
    groups = [np.flatnonzero(s.block.to_numpy() == b) for b in s.block.unique()]
    lag1 = {}
    for col in ("res_d", "res_l"):
        values = s[col].to_numpy()
        null = []
        for _ in range(reps):
            shuffled = values.copy()
            for g in groups:
                shuffled[g] = values[rng.permutation(g)]
            null.append(_lag1(shuffled, same))
        observed = _lag1(values, same)
        lag1[col] = dict(observed=float(observed), null_ci=np.percentile(null, [2.5, 97.5]).tolist(),
                         p=float((np.abs(null) >= abs(observed)).mean()))
    # The same tests on the part of log relative error the why factors (shot type) do not explain,
    # so session and timing effects are not just a proxy for which clubs were being hit.
    t["adjusted_residual"] = y - design @ coef
    adjusted = [g.to_numpy() for _, g in t.groupby("session").adjusted_residual]

    def test(stat, *args):
        return dict(zip(("statistic", "p"), map(float, stat(*args))))

    when = dict(sessions=test(kruskal, *sessions), sessions_adjusted=test(kruskal, *adjusted),
                hour=test(spearmanr, t.local_hour, t.rel_error),
                hour_adjusted=test(spearmanr, t.local_hour, t.adjusted_residual),
                position=test(spearmanr, t.position_in_block, t.rel_error),
                position_adjusted=test(spearmanr, t.position_in_block, t.adjusted_residual),
                lag1=lag1)

    summary = dict(n=len(t), why=why, r2=float(r2), worst_decile=profile, cases=cases, when=when)
    t.to_csv(ROOT / "outputs" / "error_analysis.csv", index=False)
    (ROOT / "outputs" / "error_analysis.json").write_text(json.dumps(summary, indent=2))

    print(f"why (relative landing error, n={len(t)}, adjusted R² {r2:.2f}):")
    for f, w in why.items():
        print(f"  {w['label']:36s} rho {w['rho']:+.2f} [{w['rho_ci'][0]:+.2f}, {w['rho_ci'][1]:+.2f}]   "
              f"{w['pct_per_sd']:+6.1f}% per SD [{w['pct_per_sd_ci'][0]:+.1f}, {w['pct_per_sd_ci'][1]:+.1f}]")
    print("what: worst decile vs the rest (medians; upper deck is a share):")
    for f, p in profile.items():
        print(f"  {f:18s} {p['worst']:9.3f} vs {p['rest']:9.3f}")
    print("cases:", cases)
    print("when:", json.dumps(when, indent=1))


if __name__ == "__main__":
    studies = {"explore": explore, "baselines": baselines, "neighbours": neighbours, "hybrid": hybrid,
               "error_analysis": error_analysis}
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("study", choices=studies)
    train, _ = load()
    studies[parser.parse_args().study](train)


### `src/experiments/physics_studies.py`
Physics studies; the aerodynamic diagnostic feeds several figures.

In [ ]:
%%writefile src/experiments/physics_studies.py
"""Physics studies: how far pure physics gets, and what shape the aerodynamics should take.

    python -m experiments.physics_studies inversion      checkpoint-only inversion: spin prior, wind, oracle spin
    python -m experiments.physics_studies aerodynamics   per-shot drag and lift multipliers from the full flight

Both read outputs/calibration.json from `python -m flight.calibration`. Run from src/ or with PYTHONPATH=src.
"""
import argparse
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

from data.dataset import ROOT, load
from data.features import feature_frame
from flight.inversion import RECONCILE_PARAMS, fly, invert, load_calibration, reconcile, scaled_aero, wind_for
from flight.simulator import RADIUS, RPM_TO_RAD, simulate
from modelling.evaluation import errors, mean_scales, summarise
from modelling.pipeline import lgbm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)


def inversion(train, cal):
    x = feature_frame(train)
    scales = mean_scales(train)
    shots = pd.read_csv(ROOT / "outputs" / "calibration_shots.csv")
    spin_oof = np.zeros(len(train))
    for fit, val in KFold(5, shuffle=True, random_state=42).split(x):
        spin_oof[val] = lgbm().fit(x.iloc[fit], train.launch_spin_rate.iloc[fit]).predict(x.iloc[val])

    n = len(train)
    nuisance_mean = [0.0] + [shots[k].mean() for k in ("dd", "dl", "dh")]
    nuisance_sd = [0.3] + [shots[k].std() for k in ("dd", "dl", "dh")]
    wind = wind_for(train, cal["wind"])
    loose = (0.015, 0.15, 0.3)
    variants = {
        "ML spin prior, tight checkpoints": (spin_oof, 0.9, wind, (0.01, 0.1, 0.1)),
        "ML spin prior": (spin_oof, 0.9, wind, loose),
        "ML spin prior, no wind": (spin_oof, 0.9, np.zeros_like(wind), loose),
        "ML spin fixed": (spin_oof, 0.01, wind, loose),
        "true spin fixed (oracle)": (train.launch_spin_rate.to_numpy(), 0.01, wind, loose),
    }
    rows = {}
    for name, (spin, spin_sd, w, sigma) in variants.items():
        prior_mean = np.column_stack([spin / 1000] + [np.full(n, m) for m in nuisance_mean])
        prior_sd = np.column_stack([np.full(n, spin_sd)] + [np.full(n, s) for s in nuisance_sd])
        theta, _ = invert(train, cal["aero"], w, prior_mean, prior_sd, sigma_cp=sigma)
        pred = fly(train, theta, cal["aero"], w)
        rows[name] = summarise(errors(train, pred), scales)
        rows[name]["landing_d_bias"] = (pred.landing_d - train.landing_d).mean()
    print(pd.DataFrame(rows).T.round(3))


def aerodynamics(train, cal):
    spin = train.launch_spin_rate.to_numpy()
    wind = wind_for(train, cal["wind"])
    params = reconcile(train, train, spin, cal["aero"], wind, prior_sd=np.array([1.0, 1.0, 0.5, 1.0, 0.5, 0.5]))
    log_cd, log_cl, tilt, dd, dl, dh = params[RECONCILE_PARAMS].to_numpy().T
    o = simulate(train[["vd", "vl", "vh"]].to_numpy(), spin, tilt, scaled_aero(cal["aero"], log_cd, log_cl), wind=wind)
    for k, shift in [("apex_d", dd), ("apex_h", dh), ("landing_d", dd), ("landing_l", dl)]:
        print(f"post-fit rms {k:10s} {np.sqrt(np.nanmean((o[k] + shift - train[k]) ** 2)):.3f} m")

    base = cal["aero"]
    d = params.assign(cd_mult=np.exp(log_cd), cl_mult=np.exp(log_cl), S0=RADIUS * spin * RPM_TO_RAD / train.speed)
    d["cd_eff"] = d.cd_mult * (base["cd0"] + base["cd1"] * d.S0)
    d["cl_eff"] = d.cl_mult * base["cl_max"] * d.S0 / (d.S0 + base["s_half"])
    for col in ["launch_spin_rate", "speed", "vla"]:
        print(d.groupby(pd.qcut(train[col], 6), observed=True)[["cd_mult", "cl_mult", "cd_eff", "cl_eff", "dd", "dh"]]
              .median().round(3))
    d.assign(track_id=train.track_id.values).to_csv(ROOT / "outputs" / "aero_diag.csv", index=False)


if __name__ == "__main__":
    studies = {"inversion": inversion, "aerodynamics": aerodynamics}
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("study", choices=studies)
    train, _ = load()
    studies[parser.parse_args().study](train, load_calibration())


### `src/report/style.py`
Figure styling.

In [ ]:
%%writefile src/report/style.py
"""Shared matplotlib styling for report figures: light surface, recessive chrome, fixed series order."""
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from data.dataset import ROOT

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
             "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("blue_seq", BLUE_RAMP[2:])
FIG_DIR = ROOT / "figures"


def apply():
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "font.family": "sans-serif", "font.size": 9.5,
        "text.color": INK, "axes.labelcolor": INK2, "axes.titlecolor": INK,
        "axes.titlesize": 10.5, "axes.titlelocation": "left", "figure.titlesize": 12,
        "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
        "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "lines.linewidth": 1.5, "legend.frameon": False, "legend.fontsize": 9,
        "axes.prop_cycle": mpl.cycler(color=SERIES),
    })


def save(fig, name):
    FIG_DIR.mkdir(exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


### `src/report/animation.py`
Animated launch-to-rest trajectories.

In [ ]:
%%writefile src/report/animation.py
"""Animated full trajectories: tracked flight to the net, modelled flight beyond it, bounce and roll.

    python -m report.animation [tag]    renders figures/test_shot.gif and figures/test_shots_mix.gif

Run from src/ or with PYTHONPATH=src, after `python -m modelling.pipeline --submit`.
"""
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
from matplotlib.lines import Line2D

from data.dataset import POINTS, ROOT, load
from flight.ground import bounce_and_roll
from flight.inversion import RECONCILE_PARAMS, scaled_aero, wind_for
from flight.simulator import RPM_TO_RAD, simulate
from report import style

PHASES = {"tracked": (style.SERIES[0], "tracked up to the net"),
          "modelled": (style.SERIES[1], "modelled beyond the net"),
          "ground": (style.SERIES[2], "bounce and roll")}
GRASS, NET_HEIGHT = "#e9efdf", 30.0


def trajectories(df, params, spin_rpm, aero, wind, dt=0.02):
    """Launch-to-rest path of each shot in its bay's frame: d downrange, l lateral (+ left), z above ground."""
    log_cd, log_cl, tilt, dd, dl, dh = params[RECONCILE_PARAMS].to_numpy().T
    z0 = df.launch_z.to_numpy()
    o = simulate(df[["vd", "vl", "vh"]].to_numpy(), spin_rpm, tilt, scaled_aero(aero, log_cd, log_cl),
                 wind=wind, dt=dt, keep_path=True, land_h=-z0 - dh)
    shots = []
    for i, row in enumerate(df.itertuples()):
        live = o["path_t"] < o["landing_t"][i]
        impact = np.array([o["landing_d"][i] + dd[i], o["landing_l"][i] + dl[i], 0.0])
        air = np.vstack([o["path"][live, i] + [dd[i], dl[i], dh[i] + z0[i]], impact])
        t_air = np.append(o["path_t"][live], o["landing_t"][i])
        rest = bounce_and_roll(impact, o["landing_vel"][i], o["landing_spin"][i] * RPM_TO_RAD * o["axis"][i])
        top = int(np.argmax(air[:, 2]))
        shots.append(dict(
            t_air=t_air, air=air, t_ground=t_air[-1] + rest["t"], ground=rest["path"], net_d=row.cp4_d,
            tee_z=z0[i], apex=air[top], t_apex=t_air[top], spin=float(spin_rpm[i]),
            checkpoints=np.array([[getattr(row, f"{p}_d"), getattr(row, f"{p}_l"), getattr(row, f"{p}_h") + z0[i]]
                                  for p in POINTS])))
    return shots


def test_trajectories(ids, tag="gp_feature"):
    """Trajectories for the given test track ids from the pipeline's saved outputs."""
    _, test = load()
    pred = pd.read_csv(ROOT / "outputs" / "test_predictions_local.csv").set_index("track_id")
    params = pd.read_csv(ROOT / "outputs" / "test_trajectory_params.csv").set_index("track_id")
    final = json.loads((ROOT / "outputs" / "calibration_final.json").read_text())
    final["wind"] = {int(k): v for k, v in final["wind"].items()}
    pick = test.set_index("track_id").loc[ids].reset_index()
    wind = np.zeros((len(pick), 3)) if "nowind" in tag else wind_for(pick, final["wind"])
    return trajectories(pick, params.loc[ids], pred.loc[ids].launch_spin_rate.to_numpy(), final["aero"], wind)


def _at(points, times, t):
    return np.array([np.interp(t, times, points[:, c]) for c in range(3)])


def _state(shot, t):
    """Ball position, speed and phase at time t."""
    in_air = t < shot["t_air"][-1]
    pts, times = (shot["air"], shot["t_air"]) if in_air else (shot["ground"], shot["t_ground"])
    pos = _at(pts, times, t)
    ahead = min(t + 0.02, times[-1])
    speed = np.linalg.norm(_at(pts, times, ahead) - pos) / max(ahead - t, 1e-9) if ahead > t else 0.0
    phase = ("tracked by radar" if pos[0] < shot["net_d"] else "modelled flight") if in_air else "bounce and roll"
    return pos, speed, phase


def _segments(shot, t):
    seen = shot["air"][:np.searchsorted(shot["t_air"], t, side="right")]
    cut = int(np.searchsorted(seen[:, 0], shot["net_d"]))
    on_ground = t >= shot["t_air"][-1]
    ground = shot["ground"][:np.searchsorted(shot["t_ground"], t, side="right")] if on_ground else shot["ground"][:0]
    return {"tracked": seen[:cut + 1], "modelled": seen[cut:] if cut < len(seen) else seen[:0], "ground": ground}


def _scenery(ax3, d_max, l_lim, z_max, net_d, tee_heights):
    d_grid, l_grid = np.meshgrid([0, d_max], [-l_lim, l_lim])
    ax3.plot_surface(d_grid, l_grid, np.zeros_like(d_grid), color=GRASS, alpha=1.0, shade=False, zorder=0)
    for m in range(50, int(d_max) + 1, 50):
        ax3.plot([m, m], [-l_lim, l_lim], [0, 0], color=style.AXIS, lw=0.7, zorder=1)
    width = 0.8 * l_lim
    net_l, net_z = np.meshgrid([-width, width], [0, NET_HEIGHT])
    ax3.plot_surface(np.full_like(net_l, net_d), net_l, net_z, color=style.MUTED, alpha=0.15, shade=False, zorder=1)
    for l in (-width, width):
        ax3.plot([net_d, net_d], [l, l], [0, NET_HEIGHT], color=style.INK2, lw=1.2, zorder=2)
    ax3.plot([net_d, net_d], [-width, width], [NET_HEIGHT, NET_HEIGHT], color=style.INK2, lw=0.8, zorder=2)
    for z in {round(h, 2) for h in tee_heights if h > 1}:  # upper-deck bay
        ax3.plot([0, 0], [0, 0], [0, z], color=style.INK2, lw=2, zorder=2)
        ax3.plot([-3, 3, 3, -3, -3], [-3, -3, 3, 3, -3], [z] * 5, color=style.INK2, lw=1, zorder=2)
    ax3.set(xlim=(0, d_max), ylim=(-l_lim, l_lim), zlim=(0, z_max))
    ax3.set_xlabel("downrange (m)", labelpad=6)
    ax3.set_ylabel("left of target (m)", labelpad=2)
    ax3.set_zlabel("height (m)", labelpad=0)
    ax3.set_xticks(np.arange(0, d_max + 1, 50))
    ax3.set_yticks([-l_lim // 10 * 10, 0, l_lim // 10 * 10])
    ax3.set_zticks(np.arange(0, z_max + 1, 10))
    ax3.tick_params(labelsize=7, pad=0)
    ax3.set_box_aspect((3.2, 1.0, 0.8), zoom=1.2)
    ax3.set_proj_type("persp", focal_length=0.6)
    for axis in (ax3.xaxis, ax3.yaxis, ax3.zaxis):
        axis.set_pane_color((1, 1, 1, 0))
    ax3.grid(False)


def animate(shots, title, fps=20, hold=2.0, rotate=True):
    """Build the animation; save it with `save_animation` or show it in a notebook via to_jshtml()."""
    t_end = max(s["t_ground"][-1] for s in shots)
    frames = np.arange(0.0, t_end + hold, 1.0 / fps)
    pts = np.vstack([np.vstack([s["air"], s["ground"]]) for s in shots])
    d_max = max(100.0, np.ceil(pts[:, 0].max() * 1.06 / 25) * 25)
    l_lim = max(20.0, np.ceil(np.abs(pts[:, 1]).max() * 1.25 / 10) * 10)
    z_max = max(NET_HEIGHT, np.ceil(pts[:, 2].max() * 1.15 / 10) * 10)
    net_d = float(np.mean([s["net_d"] for s in shots]))
    labelled = len(shots) == 1

    fig = plt.figure(figsize=(13, 6.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[2.1, 1], hspace=0.5, wspace=0.3, left=0.0, right=0.98)
    ax3 = fig.add_subplot(grid[:, 0], projection="3d", computed_zorder=False)
    side, top = fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, 1])
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    _scenery(ax3, d_max, l_lim, z_max, net_d, [s["tee_z"] for s in shots])

    for ax, ylim, ylabel, name in [(side, (0, z_max), "height (m)", "side view"),
                                   (top, (-l_lim, l_lim), "left of target (m)", "from above")]:
        ax.set(xlim=(0, d_max), ylim=ylim, ylabel=ylabel)
        ax.set_title(name)
        ax.axvline(net_d, color=style.INK2, lw=1.5)
    top.set_xlabel("downrange (m)")
    side.text(net_d + 2, z_max * 0.9, "net", color=style.INK2, fontsize=8)

    ball = dict(ms=6, mfc="white", mec=style.INK, mew=1.2, ls="", zorder=5)
    artists = []
    for s in shots:
        a = {ph: (ax3.plot([], [], [], color=c, lw=2, zorder=3)[0], side.plot([], [], color=c, lw=1.5)[0],
                  top.plot([], [], color=c, lw=1.5)[0]) for ph, (c, _) in PHASES.items()}
        a["shadow"] = ax3.plot([], [], [], "o", ms=4, color=style.MUTED, alpha=0.5, zorder=2)[0]
        a["ball"] = (ax3.plot([], [], [], "o", **ball)[0], side.plot([], [], "o", **ball)[0],
                     top.plot([], [], "o", **ball)[0])
        ax3.plot(*s["checkpoints"].T, "o", ms=3, color=style.SERIES[0], zorder=4)
        side.plot(*s["checkpoints"][:, [0, 2]].T, "o", ms=3, color=style.SERIES[0])
        note = dict(textcoords="offset points", ha="center", fontsize=8, color=style.INK2, visible=False)
        a["apex"] = side.annotate(f"apex {s['apex'][2]:.0f} m", s["apex"][[0, 2]], xytext=(0, 6), **note)
        a["carry"] = top.annotate(f"carry {s['air'][-1][0]:.0f} m", s["air"][-1][[0, 1]], xytext=(0, 8), **note)
        a["rest"] = top.annotate(f"rest {s['ground'][-1][0]:.0f} m", s["ground"][-1][[0, 1]], xytext=(0, -14), **note)
        artists.append(a)
    info = ax3.text2D(0.03, 0.9, "", transform=ax3.transAxes, color=style.INK, fontsize=9.5, va="top")
    ax3.legend(handles=[Line2D([], [], color=c, lw=2, label=label) for c, label in PHASES.values()],
               loc="lower left", bbox_to_anchor=(0.02, 0.04), fontsize=8.5)

    def update(k):
        t = frames[k]
        for s, a in zip(shots, artists):
            for ph, seg in _segments(s, t).items():
                line3, line_side, line_top = a[ph]
                line3.set_data_3d(seg[:, 0], seg[:, 1], seg[:, 2])
                line_side.set_data(seg[:, 0], seg[:, 2])
                line_top.set_data(seg[:, 0], seg[:, 1])
            pos, speed, phase = _state(s, t)
            a["ball"][0].set_data_3d([pos[0]], [pos[1]], [pos[2]])
            a["ball"][1].set_data([pos[0]], [pos[2]])
            a["ball"][2].set_data([pos[0]], [pos[1]])
            a["shadow"].set_data_3d([pos[0]], [pos[1]], [0.0])
            if labelled:
                a["apex"].set_visible(t >= s["t_apex"])
                a["carry"].set_visible(t >= s["t_air"][-1])
                a["rest"].set_visible(t >= s["t_ground"][-1])
                info.set_text(f"{t:4.1f} s  ·  {phase}\n{speed:4.1f} m/s   height {pos[2]:4.1f} m   "
                              f"downrange {pos[0]:5.1f} m\nlaunch spin {s['spin']:,.0f} rpm")
        if rotate:
            ax3.view_init(elev=14, azim=-58 + 22 * t / frames[-1])
        return []

    return FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps)


def save_animation(anim, path, fps=20, dpi=90):
    writer = PillowWriter(fps=fps) if str(path).endswith(".gif") else FFMpegWriter(fps=fps, bitrate=2400)
    anim.save(path, writer=writer, dpi=dpi)
    plt.close(anim._fig)


def render_submission(tag="gp_feature", fps=20):
    """Render a single test shot and a mix of five test shots from the pipeline's saved outputs."""
    spins = pd.read_csv(ROOT / "outputs" / "test_predictions_local.csv").set_index("track_id").launch_spin_rate
    single = [(spins - spins.quantile(0.2)).abs().idxmin()]
    save_animation(animate(test_trajectories(single, tag), "A test shot: tracked to the net, predicted all the way to rest"),
                   ROOT / "figures" / "test_shot.gif", fps)
    mix = [(spins - spins.quantile(q)).abs().idxmin() for q in (0.05, 0.3, 0.55, 0.8, 0.97)]
    save_animation(animate(test_trajectories(mix, tag), "Five test shots, from low-spin to wedge-like"),
                   ROOT / "figures" / "test_shots_mix.gif", fps)


if __name__ == "__main__":
    import sys

    style.apply()
    render_submission(sys.argv[1] if len(sys.argv) > 1 else "gp_feature")


### `src/report/figures.py`
Report figures 01-15.

In [ ]:
%%writefile src/report/figures.py
"""Report figures: exploratory analysis (`eda`) and model results and error analysis (`results`).

    python -m report.figures eda                 figures 01-05
    python -m report.figures results [--tag T]   figures 06-15 (needs the pipeline's CV and submission outputs
                                                 and `python -m experiments.ml_studies error_analysis`)

Run from src/ or with PYTHONPATH=src.
"""
import argparse
import json

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from data.dataset import CP_PLANES, HEADING, ROOT, U, V, load
from data.features import feature_frame, kinematic_features
from flight.inversion import load_calibration, reconcile, wind_for
from flight.simulator import coefficients, simulate
from modelling.evaluation import errors
from report import style
from report.animation import PHASES, test_trajectories, trajectories

style.apply()
train, test = load()
cal = load_calibration()


def fig_geometry():
    fig, ax = plt.subplots(figsize=(7.5, 6.0))
    lat = np.array([-25.0, 20.0])
    for k, s in enumerate(CP_PLANES):
        p = s * U[None, :] + lat[:, None] * V[None, :]
        net = k == 3
        ax.plot(p[:, 0], p[:, 1], color=style.INK2 if net else style.AXIS, lw=2 if net else 1,
                ls="-" if net else "--", zorder=1)
        ax.annotate("net" if net else f"cp{k + 1}", p[0], xytext=(0, -12), textcoords="offset points",
                    ha="center", color=style.INK2, fontsize=8.5)
    sc = ax.scatter(train.landing_x, train.landing_y, c=train.launch_spin_rate, cmap=style.SEQ,
                    s=18, lw=0, zorder=2)
    tees = pd.concat([train, test]).groupby("tee")[["launch_x", "launch_y", "launch_z"]].first()
    ax.scatter(tees.launch_x, tees.launch_y, marker="s", s=40, color=style.SERIES[1], zorder=3)
    raised = tees[tees.launch_z > 1].iloc[0]
    ax.annotate("four bays (one on an\nupper deck, 4.1 m up)", (raised.launch_x, raised.launch_y),
                xytext=(-10, -34), textcoords="offset points", ha="left", color=style.INK2, fontsize=8.5)
    start = tees[["launch_x", "launch_y"]].mean().to_numpy()
    ax.annotate("", xy=start + 250 * U, xytext=start,
                arrowprops=dict(arrowstyle="-|>", color=style.MUTED, lw=1), zorder=0)
    ax.text(*(start + 245 * U + 5 * V), "downrange, 28.4° from x", color=style.INK2, fontsize=8.5,
            rotation=np.degrees(HEADING), rotation_mode="anchor", ha="right", va="bottom", zorder=4,
            bbox=dict(boxstyle="round,pad=0.25", fc=style.SURFACE, ec="none", alpha=0.9))
    fig.colorbar(sc, ax=ax, shrink=0.75, pad=0.02).set_label("launch spin (rpm)", color=style.INK2)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("Range layout: training landings, coloured by spin")
    style.save(fig, "01_range_geometry")


def fig_side_view(n=28, seed=3):
    diag = pd.read_csv(ROOT / "outputs" / "aero_diag.csv").set_index("track_id")
    sub = train.sample(n, random_state=seed)
    d = diag.loc[sub.track_id]
    base = cal["aero"]
    aero = dict(base, cd0=base["cd0"] * d.cd_mult.to_numpy(), cd1=base["cd1"] * d.cd_mult.to_numpy(),
                cl_max=base["cl_max"] * d.cl_mult.to_numpy())
    o = simulate(sub[["vd", "vl", "vh"]].to_numpy(), sub.launch_spin_rate.to_numpy(), d.tilt.to_numpy(),
                 aero, wind=wind_for(sub, cal["wind"]), keep_path=True)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    net = train.cp4_d.mean()
    ax.axvspan(net, 265, color=style.GRID, alpha=0.45, lw=0, zorder=0)
    ax.axvline(net, color=style.INK2, lw=2, zorder=1)
    ax.text(net + 3, 46, "net: everything to the right\nis hidden on a netted range", color=style.INK2,
            fontsize=8.5, va="top")
    for i in range(n):
        live = o["path_t"] <= o["landing_t"][i]
        dd = o["path"][live, i, 0] + d.dd.iloc[i]
        hh = o["path"][live, i, 2] + d.dh.iloc[i]
        seen = dd <= sub.cp4_d.iloc[i]
        ax.plot(dd[seen], hh[seen], color=style.SERIES[0], lw=1.3, zorder=2)
        ax.plot(dd[~seen], hh[~seen], color=style.BLUE_RAMP[3], lw=1.0, zorder=2)
    ax.scatter(sub.apex_d, sub.apex_h, s=22, facecolor="none", edgecolor=style.INK2, lw=1, zorder=3)
    ax.scatter(sub.landing_d, sub.landing_h, s=26, marker="v", color=style.SERIES[1], lw=0, zorder=3)
    frac_d = (train.cp4_d / train.landing_d).mean()
    frac_t = (train.cp4_t / train.landing_t).mean()
    ax.set_xlim(0, 265)
    ax.set_ylim(0, 50)
    ax.set_xlabel("downrange from the bay (m)")
    ax.set_ylabel("height above tee (m)")
    ax.legend(handles=[Line2D([], [], color=style.SERIES[0], lw=1.3, label="seen before the net"),
                       Line2D([], [], color=style.BLUE_RAMP[3], lw=1, label="flight behind the net"),
                       Line2D([], [], marker="o", ls="", mfc="none", mec=style.INK2, label="measured apex"),
                       Line2D([], [], marker="v", ls="", color=style.SERIES[1], label="measured landing")],
              loc="upper right", ncols=2)
    ax.set_title(f"The net sees {frac_d:.0%} of the carry and {frac_t:.0%} of the flight time "
                 f"({n} random training shots)")
    style.save(fig, "02_what_the_net_hides")


def fig_train_test():
    both = pd.concat([feature_frame(train), feature_frame(test)])
    y = np.r_[np.zeros(len(train)), np.ones(len(test))]
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, n_jobs=4, verbose=-1)
    p = cross_val_predict(clf, both, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                          method="predict_proba")[:, 1]
    auc = roc_auc_score(y, p)
    cols = {"launch speed (m/s)": "speed", "vertical launch angle (°)": "vla",
            "horizontal launch angle (°)": "hla", "time to reach the net (s)": "cp4_t"}
    fig, axes = plt.subplots(1, 4, figsize=(12, 2.9))
    for ax, (label, col) in zip(axes, cols.items()):
        bins = np.histogram_bin_edges(pd.concat([train[col], test[col]]), 28)
        ax.hist(train[col], bins, density=True, histtype="step", lw=1.5, color=style.SERIES[0],
                label=f"train ({len(train)})")
        ax.hist(test[col], bins, density=True, histtype="step", lw=1.5, color=style.SERIES[1],
                label=f"test ({len(test)})")
        ax.set_xlabel(label)
        ax.set_yticks([])
        ax.grid(axis="y", visible=False)
    axes[0].legend(loc="upper left")
    fig.suptitle(f"Train and test come from the same sessions: a classifier can barely tell them apart "
                 f"(AUC {auc:.2f})", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "03_train_vs_test")
    return auc


def fig_spin_fingerprint():
    kin = kinematic_features(train)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
    for ax, col, label in [(axes[0], "clu_mid", "lift coefficient measured between checkpoints"),
                           (axes[1], "cd_mid", "drag coefficient measured between checkpoints")]:
        ax.scatter(kin[col], train.launch_spin_rate, s=14, color=style.SERIES[0], alpha=0.55, lw=0)
        r = np.corrcoef(kin[col], train.launch_spin_rate)[0, 1]
        ax.text(0.03, 0.95, f"r = {r:.2f}", transform=ax.transAxes, va="top", color=style.INK2)
        ax.set_xlabel(label)
    axes[0].set_ylabel("launch-monitor spin (rpm)")
    fig.suptitle("Spin leaves a fingerprint in the first 60 m", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "04_spin_fingerprint")


def fig_session():
    both = pd.concat([train.assign(split="train"), test.assign(split="test")]).sort_values("launch_time")
    s = train.sort_values("launch_time")
    same = s.block.eq(s.block.shift())
    lag = np.corrcoef(s.launch_spin_rate[same], s.launch_spin_rate.shift()[same])[0, 1]
    blk = train.block.value_counts().idxmax()
    b = both[both.block == blk]
    t0 = b.launch_time.min()
    fig, ax = plt.subplots(figsize=(9, 3.3))
    tr = b[b.split == "train"]
    ax.plot((tr.launch_time - t0) / 60, tr.launch_spin_rate, "-o", color=style.SERIES[0], ms=5, lw=1,
            label="training shot spin")
    te = b[b.split == "test"]
    ax.scatter((te.launch_time - t0) / 60, np.full(len(te), 0.03), marker="|", s=90, color=style.SERIES[1],
               transform=ax.get_xaxis_transform(), label="test shot (spin unknown)")
    ax.set_xlabel(f"minutes into the session ({pd.to_datetime(t0, unit='s'):%d %b %Y})")
    ax.set_ylabel("spin (rpm)")
    ax.legend(loc="upper right", ncols=2)
    ax.set_title(f"Players hit runs with one club: consecutive shots' spin correlates at r = {lag:.2f}")
    style.save(fig, "05_same_club_runs")
    return lag


def fig_model_comparison():
    """Landing error per approach and per ablation of the final model, from outputs/cv_summary.json."""
    summary = json.loads((ROOT / "outputs" / "cv_summary.json").read_text())
    labels, values, headings = [], [], []
    for group, models in summary.items():
        headings.append(len(labels))
        labels.append(group)
        values.append(np.nan)
        for name, metrics in models.items():
            labels.append(name)
            values.append(metrics["landing"])
    values = np.array(values)
    y = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8.5, 0.4 * len(labels) + 1.1))
    ax.barh(y, np.nan_to_num(values), height=0.62, color=style.SERIES[0])
    for yi, v in zip(y, values):
        if np.isfinite(v):
            ax.text(v + 0.06, yi, f"{v:.2f} m", va="center", color=style.INK2, fontsize=9)
    ax.set_yticks(y, labels)
    for i in headings:
        ax.get_yticklabels()[i].set_fontweight("bold")
        ax.get_yticklabels()[i].set_color(style.INK)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, np.nanmax(values) * 1.15)
    ax.set_xlabel("mean landing-position error (m), 5-fold cross-validation on train")
    ax.set_title("Approaches and ablations: what each ingredient of the final model contributes")
    style.save(fig, "06_model_comparison")


def _oof(tag):
    return pd.read_csv(ROOT / "outputs" / f"pipeline_oof_{tag}.csv").set_index("track_id").loc[train.track_id]


def fig_predicted_vs_actual(tag):
    oof = _oof(tag)
    err = errors(train, oof)
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
    for ax, col, label in [(axes[0], "landing_d", "carry downrange (m)"),
                           (axes[1], "landing_l", "landing lateral (m, + = left)")]:
        truth, pred = train[col].to_numpy(), oof[col].to_numpy()
        lo, hi = min(truth.min(), pred.min()), max(truth.max(), pred.max())
        ax.plot([lo, hi], [lo, hi], color=style.AXIS, lw=1)
        ax.scatter(truth, pred, s=12, color=style.SERIES[0], alpha=0.6, lw=0)
        ax.set_xlabel(f"measured {label}")
        ax.set_ylabel("predicted")
        ax.text(0.03, 0.95, f"mean abs error {np.abs(truth - pred).mean():.1f} m", transform=ax.transAxes,
                va="top", color=style.INK2)
    axes[2].hist(err.landing, bins=30, color=style.SERIES[0], rwidth=0.85)
    for q, label, height in [(0.5, "median", 0.95), (0.9, "90th percentile", 0.8)]:
        v = err.landing.quantile(q)
        axes[2].axvline(v, color=style.INK2, lw=1, ls="--")
        axes[2].text(v, axes[2].get_ylim()[1] * height, f" {label} {v:.1f} m", color=style.INK2, fontsize=8.5,
                     va="top")
    axes[2].set_xlabel("landing-position error (m)")
    axes[2].set_ylabel("shots")
    axes[2].grid(axis="x", visible=False)
    fig.suptitle("Out-of-fold predictions for every training shot", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "07_predicted_vs_actual")


def fig_error_by_shot(tag):
    err = errors(train, _oof(tag))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
    for ax, col, label in [(axes[0], "launch_spin_rate", "launch spin (rpm)"), (axes[1], "speed", "launch speed (m/s)")]:
        bins = pd.qcut(train[col], 6)
        stats = err.landing.groupby(bins, observed=True).quantile([0.25, 0.5, 0.75]).unstack()
        centres = train[col].groupby(bins, observed=True).median()
        ax.vlines(centres, stats[0.25], stats[0.75], color=style.BLUE_RAMP[4], lw=4)
        ax.plot(centres, stats[0.5], "o", color=style.SERIES[0], ms=7)
        ax.set_xlabel(f"{label}, sextiles")
    axes[0].set_ylabel("landing error (m)")
    fig.suptitle("Landing error by shot type (dot = median, bar = middle 50%)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "08_error_by_shot_type")


def fig_aerodynamics():
    diag = pd.read_csv(ROOT / "outputs" / "aero_diag.csv")
    a = cal["aero"]
    S = np.linspace(0.02, 0.8, 200)
    cd, cl = coefficients(S, a)
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
    for ax, pts, curve, label in [(axes[0], diag.cl_eff, cl, "lift coefficient C_L"),
                                  (axes[1], diag.cd_eff, cd, "drag coefficient C_D")]:
        ax.scatter(diag.S0, pts, s=10, color=style.BLUE_RAMP[4], alpha=0.5, lw=0, label="per-shot fit")
        ax.plot(S, curve, color=style.INK, lw=2, label="calibrated curve")
        ax.set_xlabel("spin factor at launch, S = rω / v")
        ax.set_ylabel(label)
    axes[0].legend(loc="lower right")
    fig.suptitle(f"Calibrated range-ball aerodynamics:  C_D = {a['cd0']:.3f} + {a['cd1']:.3f}·S,   "
                 f"C_L = {a['cl_max']:.3f}·S / (S + {a['s_half']:.3f})", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "09_aerodynamics")


def fig_wind():
    starts = pd.concat([train, test]).groupby("block").launch_time.min().sort_values()
    wind = pd.DataFrame(cal["wind"], index=["along", "across"]).T
    wind = wind.loc[[b for b in starts.index if b in wind.index]]
    labels = [pd.to_datetime(starts[b], unit="s").strftime("%d %b %H:%M") for b in wind.index]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
    y = np.arange(len(wind))
    for ax, col, label in [(axes[0], "along", "along the range (m/s, + = tailwind)"),
                           (axes[1], "across", "across the range (m/s, + = blowing left)")]:
        ax.axvline(0, color=style.AXIS, lw=1)
        ax.hlines(y, 0, wind[col], color=style.BLUE_RAMP[4], lw=2)
        ax.plot(wind[col], y, "o", color=style.SERIES[0], ms=6)
        ax.set_xlabel(label)
        ax.grid(axis="y", visible=False)
    mean_along = wind.along.mean()
    axes[0].axvline(mean_along, color=style.INK2, lw=1, ls="--")
    axes[0].set_xlabel(f"along the range (m/s, + = tailwind)\ndashed: mean {mean_along:.1f} m/s, an offset that "
                       "absorbs unmodelled drag")
    axes[0].set_yticks(y, labels)
    axes[0].invert_yaxis()
    fig.suptitle("Effective wind per hitting block, fitted from training landings: the spread between blocks "
                 "is the weather", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "10_session_wind")


def fig_test_flights(tag, n=40):
    pred = pd.read_csv(ROOT / "outputs" / "test_predictions_local.csv").set_index("track_id").loc[test.track_id]
    shots = test_trajectories(test.track_id.sample(n, random_state=1).tolist(), tag)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw=dict(width_ratios=[1, 2.1]))
    sc = ax1.scatter(-pred.landing_l, pred.landing_d, c=pred.launch_spin_rate, cmap=style.SEQ, s=12, lw=0)
    ax1.set(xlabel="right of target (m)", ylabel="downrange (m)", aspect="equal")
    fig.colorbar(sc, ax=ax1, shrink=0.85).set_label("predicted spin (rpm)", color=style.INK2)
    ax1.set_title(f"Predicted landings, all {len(test)} test shots")
    ax2.axvline(test.cp4_d.mean(), color=style.INK2, lw=1.5)
    for s in shots:
        cut = int(np.searchsorted(s["air"][:, 0], s["net_d"]))
        ax2.plot(s["air"][:cut + 1, 0], s["air"][:cut + 1, 2], color=PHASES["tracked"][0], lw=1.2)
        ax2.plot(s["air"][cut:, 0], s["air"][cut:, 2], color=PHASES["modelled"][0], lw=1, alpha=0.85)
        ax2.plot(s["ground"][:, 0], s["ground"][:, 2], color=PHASES["ground"][0], lw=1)
    ax2.legend(handles=[Line2D([], [], color=c, lw=2, label=label) for c, label in PHASES.values()], loc="upper right")
    ax2.set(xlabel="downrange (m)", ylabel="height above ground (m)", ylim=(0, None), xlim=(0, None))
    ax2.set_title(f"Full predicted flights for {n} random test shots")
    fig.tight_layout()
    style.save(fig, "11_test_flights")


def fig_bounce_and_roll(tag):
    spins = pd.read_csv(ROOT / "outputs" / "test_predictions_local.csv").set_index("track_id").launch_spin_rate
    ids = [(spins - spins.quantile(q)).abs().idxmin() for q in (0.03, 0.35, 0.7, 0.97)]
    shots = test_trajectories(ids, tag)
    fig, axes = plt.subplots(len(shots), 1, figsize=(9, 6.5), sharex=True)
    for ax, s in zip(axes, shots):
        land = s["air"][-1]
        v = (s["air"][-1] - s["air"][-2]) / (s["t_air"][-1] - s["t_air"][-2])
        angle = np.degrees(np.arctan2(-v[2], np.hypot(v[0], v[1])))

        def along(p):
            return np.sign(p[:, 0] - land[0]) * np.hypot(p[:, 0] - land[0], p[:, 1] - land[1])

        tail = s["air"][s["air"][:, 0] >= land[0] - 25]
        ax.axhline(0, color=style.AXIS, lw=1)
        ax.plot(along(tail), tail[:, 2], color=style.SERIES[1], lw=1.5)
        ax.plot(along(s["ground"]), s["ground"][:, 2], color=style.SERIES[2], lw=1.8)
        run = along(s["ground"][-1:])[0]
        ax.plot([run], [0], "o", ms=6, mfc="white", mec=style.INK)
        ax.text(0.01, 0.92, f"{s['spin']:,.0f} rpm · lands at {np.linalg.norm(v):.0f} m/s and {angle:.0f}° · "
                f"comes to rest {run:+.1f} m from the pitch mark", transform=ax.transAxes, va="top",
                color=style.INK2, fontsize=9)
        ax.set_ylim(0, 3)
        ax.grid(axis="x", visible=False)
    axes[-1].set_xlabel("distance from the landing point (m)")
    fig.supylabel("height (m)", color=style.INK2, fontsize=9.5)
    fig.suptitle("Bounce and roll: low-spin shots release, high-spin shots check and spin back", x=0.01,
                 ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "12_bounce_and_roll")


def _error_analysis():
    return (json.loads((ROOT / "outputs" / "error_analysis.json").read_text()),
            pd.read_csv(ROOT / "outputs" / "error_analysis.csv").set_index("track_id"))


def fig_case_studies():
    """What: measured and predicted flights for the best, a typical and the worst out-of-fold shots."""
    summary, table = _error_analysis()
    oof = _oof("gp_feature")
    diag = pd.read_csv(ROOT / "outputs" / "aero_diag.csv").set_index("track_id")
    ids = list(summary["cases"].values())
    pick = train.set_index("track_id").loc[ids].reset_index()
    wind = wind_for(pick, cal["wind"])
    spin_pred = oof.loc[ids].launch_spin_rate.to_numpy()
    measured = trajectories(pick, diag.loc[ids], pick.launch_spin_rate.to_numpy(), cal["aero"], wind)
    predicted = trajectories(pick, reconcile(pick, oof.loc[ids], spin_pred, cal["aero"], wind), spin_pred,
                             cal["aero"], wind)

    fig, axes = plt.subplots(len(ids), 2, figsize=(12, 2.6 * len(ids) + 0.6), gridspec_kw=dict(width_ratios=[1.6, 1]))
    for row, (case, tid) in enumerate(summary["cases"].items()):
        shot, m, p, info, d = pick.iloc[row], measured[row], predicted[row], table.loc[tid], diag.loc[tid]
        side, top = axes[row]
        cut = int(np.searchsorted(p["air"][:, 0], p["net_d"]))
        for ax, j in ((side, 2), (top, 1)):
            ax.axvline(shot.cp4_d, color=style.AXIS, lw=1)
            ax.plot(m["air"][:, 0], m["air"][:, j], color=style.INK2, lw=1.2, ls="--")
            ax.plot(p["air"][:cut + 1, 0], p["air"][:cut + 1, j], color=style.SERIES[0], lw=1.8)
            ax.plot(p["air"][cut:, 0], p["air"][cut:, j], color=style.SERIES[1], lw=1.8)
            ax.plot(p["checkpoints"][:, 0], p["checkpoints"][:, j], "o", ms=3.5, color=style.SERIES[0])
        z0 = shot.launch_z
        side.plot(shot.apex_d, shot.apex_h + z0, "o", mfc="none", mec=style.INK2, ms=7)
        side.plot(shot.landing_d, z0, "v", color=style.INK2, ms=7)
        side.plot(oof.loc[tid].landing_d, z0, "v", color=style.SERIES[1], ms=7)
        top.plot(shot.landing_d, shot.landing_l, "v", color=style.INK2, ms=7)
        top.plot(oof.loc[tid].landing_d, oof.loc[tid].landing_l, "v", color=style.SERIES[1], ms=7)
        side.set_title(f"{case}: landing off by {info.landing_error:.1f} m ({info.rel_error:.1f}% of carry)"
                       f"{', upper-deck bay' if shot.elevated else ''}")
        side.text(0.01, 0.95, f"spin {shot.launch_spin_rate:,.0f} rpm, predicted {oof.loc[tid].launch_spin_rate:,.0f}\n"
                  f"{shot.speed:.0f} m/s at {shot.vla:.0f}° · this ball: drag ×{d.cd_mult:.2f}, lift ×{d.cl_mult:.2f}, "
                  f"tilt {np.degrees(d.tilt):+.0f}°", transform=side.transAxes, va="top", fontsize=8.5, color=style.INK2)
        side.set(ylabel="height (m)", xlim=(0, None), ylim=(0, max(m["air"][:, 2].max(), p["air"][:, 2].max()) * 1.45))
        top.set(ylabel="left of target (m)", xlim=(0, None))
    axes[-1, 0].set_xlabel("downrange (m)")
    axes[-1, 1].set_xlabel("downrange (m)")
    fig.legend(handles=[Line2D([], [], color=style.INK2, lw=1.2, ls="--", label="measured flight"),
                        Line2D([], [], color=style.SERIES[0], lw=1.8, label="tracked to the net"),
                        Line2D([], [], color=style.SERIES[1], lw=1.8, label="predicted beyond the net"),
                        Line2D([], [], marker="o", ls="", mfc="none", mec=style.INK2, label="measured apex"),
                        Line2D([], [], marker="v", ls="", color=style.INK2, label="measured landing"),
                        Line2D([], [], marker="v", ls="", color=style.SERIES[1], label="predicted landing")],
               loc="upper right", ncols=3, bbox_to_anchor=(0.99, 1.0), fontsize=8.5)
    fig.suptitle("What the errors look like: measured and predicted flights", x=0.01, y=0.995, ha="left",
                 fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    style.save(fig, "13_case_studies")


def fig_error_attribution():
    """Why: association of shot-level factors with the relative landing error."""
    summary, _ = _error_analysis()
    why = summary["why"]
    order = sorted(why, key=lambda f: abs(why[f]["pct_per_sd"]))
    y = np.arange(len(order))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), sharey=True)
    for ax, key, title, xlabel in [
            (axes[0], "rho", "Univariate", "Spearman rank correlation with relative error"),
            (axes[1], "pct_per_sd", "Adjusted for all other factors", "change in relative error per 1 SD of the factor (%)")]:
        ax.axvline(0, color=style.AXIS, lw=1)
        ax.hlines(y, [why[f][f"{key}_ci"][0] for f in order], [why[f][f"{key}_ci"][1] for f in order],
                  color=style.BLUE_RAMP[4], lw=3)
        ax.plot([why[f][key] for f in order], y, "o", color=style.SERIES[0], ms=7)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.grid(axis="y", visible=False)
    axes[0].set_yticks(y, [why[f]["label"] for f in order])
    fig.suptitle(f"Why shots miss: {why[order[-1]]['label']} has the largest adjusted effect on relative landing error",
                 x=0.01, ha="left", fontweight="bold")
    fig.text(0.01, -0.02, f"n = {summary['n']} out-of-fold shots; relative error = landing error ÷ carry; bars are "
             f"bootstrap 95% intervals; regression R² = {summary['r2']:.2f} on log relative error.",
             color=style.INK2, fontsize=8.5)
    fig.tight_layout()
    style.save(fig, "14_error_attribution")


def _p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def fig_error_timing():
    """When: relative landing error by session, time of day and position in the hitting block, and carry-over
    between consecutive shots. Adjusted tests use the error left unexplained by the why factors (shot type)."""
    summary, table = _error_analysis()
    when = summary["when"]
    fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.2), gridspec_kw=dict(width_ratios=[1.3, 1, 1, 0.75]))

    stats = table.groupby("session").rel_error.quantile([0.25, 0.5, 0.75]).unstack().sort_index()
    x = np.arange(len(stats))
    axes[0].vlines(x, stats[0.25], stats[0.75], color=style.BLUE_RAMP[4], lw=4)
    axes[0].plot(x, stats[0.5], "o", color=style.SERIES[0], ms=7)
    axes[0].set_xticks(x, [pd.Timestamp(s).strftime("%d %b") for s in stats.index], rotation=45, ha="right")
    axes[0].set_ylabel("relative landing error (% of carry)")
    axes[0].set_title(f"By session\nKruskal–Wallis {_p(when['sessions']['p'])}; adjusted {_p(when['sessions_adjusted']['p'])}",
                      fontsize=9.5)
    axes[0].grid(axis="x", visible=False)

    axes[1].sharey(axes[2])
    axes[1].set_ylabel("relative landing error (% of carry)")
    for ax, col, width, label, key, xlabel in [
            (axes[1], "local_hour", 0.5, "By time of day", "hour", "local time (SAST, hours)"),
            (axes[2], "position_in_block", 0.1, "Through a hitting block", "position", "position in the block (0 = first, 1 = last)")]:
        ax.scatter(table[col], table.rel_error, s=8, color=style.BLUE_RAMP[3], alpha=0.5, lw=0)
        edges = np.arange(np.floor(table[col].min() / width) * width, table[col].max() + width, width)
        medians = table.groupby(pd.cut(table[col], edges), observed=True).rel_error.median()
        ax.plot([b.mid for b in medians.index], medians.to_numpy(), "-o", color=style.SERIES[0], ms=5)
        ax.set(ylim=(0, table.rel_error.quantile(0.98)), xlabel=xlabel)
        raw, adj = when[key], when[f"{key}_adjusted"]
        ax.set_title(f"{label}\nρ = {raw['statistic']:+.2f} ({_p(raw['p'])}); adjusted ρ = {adj['statistic']:+.2f} ({_p(adj['p'])})",
                     fontsize=9.5)

    for i, col in enumerate(("res_d", "res_l")):
        lag = when["lag1"][col]
        axes[3].vlines(i, *lag["null_ci"], color=style.GRID, lw=14)
        axes[3].plot(i, lag["observed"], "o", color=style.SERIES[0], ms=8)
        axes[3].text(i + 0.14, lag["observed"], _p(lag["p"]), va="center", fontsize=8.5, color=style.INK2)
    axes[3].axhline(0, color=style.AXIS, lw=1)
    axes[3].set(xlim=(-0.5, 1.9), ylabel="lag-1 correlation of residuals")
    axes[3].set_xticks([0, 1], ["downrange", "lateral"])
    axes[3].set_title("Carry-over between\nconsecutive shots", fontsize=9.5)
    axes[3].legend(handles=[Line2D([], [], color=style.GRID, lw=8, label="permutation null, 95%"),
                            Line2D([], [], marker="o", ls="", color=style.SERIES[0], label="observed")],
                   loc="lower right", fontsize=8)
    axes[3].grid(axis="x", visible=False)

    sessions = ("sessions still differ once shot type is accounted for" if when["sessions_adjusted"]["p"] < 0.05
                else "differences between sessions are explained by shot type")
    carry = ("consecutive residuals are correlated" if min(l["p"] for l in when["lag1"].values()) < 0.05
             else "consecutive residuals are independent")
    fig.suptitle(f"When shots miss: {sessions}, and {carry}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    style.save(fig, "15_error_timing")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("group", nargs="?", default="eda", choices=["eda", "results"])
    parser.add_argument("--tag", default="gp_feature")
    args = parser.parse_args()
    if args.group == "eda":
        fig_geometry()
        fig_side_view()
        auc = fig_train_test()
        fig_spin_fingerprint()
        lag = fig_session()
        print(f"adversarial AUC {auc:.3f}, lag-1 spin corr {lag:.3f}")
    else:
        fig_model_comparison()
        fig_predicted_vs_actual(args.tag)
        fig_error_by_shot(args.tag)
        fig_aerodynamics()
        fig_wind()
        fig_test_flights(args.tag)
        fig_bounce_and_roll(args.tag)
        fig_case_studies()
        fig_error_attribution()
        fig_error_timing()


## 1. Calibrate the flight model on the training shots
Fits the global drag and lift coefficients, per-shot spin-axis tilt and start offsets, and one wind vector per hitting block, all with the launch-monitor spin known. Then runs the per-shot aerodynamic diagnostic used by several figures.

In [ ]:
run("flight.calibration", tail=30)
run("experiments.physics_studies", "aerodynamics", tail=12)

## 2. Honest cross-validation and the submission
Inside every fold: calibration, block wind, the LightGBM spin model, checkpoint inversion and the Gaussian-process layer are all refitted. The final fit on all training shots writes `submission.csv` and reconciles a full trajectory for every test shot.

In [ ]:
import shutil

run("modelling.pipeline", "--cv", "--submit", tail=8)
shutil.copy(WORK / "submissions" / "submission.csv", WORK / "submission.csv")

import pandas as pd
submission = pd.read_csv(WORK / "submission.csv")
print(submission.shape)
submission.head()

## 3. Error analysis: what, why and when
Explains the out-of-fold errors: case studies and a worst-decile profile (what), rank correlations and adjusted regression effects over seven shot-level factors (why), and session, time-of-day, block-position and carry-over tests (when). Ablations of the final model are run separately with `python -m modelling.pipeline --cv --oracle-spin` and `--no-fingerprints`; their results are recorded in the summary below.

In [ ]:
run("experiments.ml_studies", "error_analysis", tail=30)

## 4. Figures
The comparison figure reads a summary of the cross-validation results. The benchmark and ablation rows come from `python -m experiments.ml_studies baselines` and the pipeline ablations, which take longer to run, so they are recorded here.

In [ ]:
%%writefile outputs/cv_summary.json
{
  "Approaches": {
    "Ridge, degree-2 features": {"landing": 5.140, "apex": 2.879, "apex_t": 0.090, "landing_t": 0.145, "spin": 1019.0, "composite": 0.167},
    "Extra trees": {"landing": 5.864, "apex": 3.320, "apex_t": 0.093, "landing_t": 0.159, "spin": 741.2, "composite": 0.167},
    "LightGBM": {"landing": 5.372, "apex": 3.114, "apex_t": 0.084, "landing_t": 0.138, "spin": 717.9, "composite": 0.154},
    "Gaussian process": {"landing": 4.687, "apex": 2.350, "apex_t": 0.077, "landing_t": 0.130, "spin": 824.1, "composite": 0.144},
    "Physics only (calibrated, session wind)": {"landing": 6.064, "apex": 3.298, "apex_t": 0.094, "landing_t": 0.201, "spin": 717.9, "composite": 0.174},
    "Hybrid: physics feature + GP (final)": {"landing": 4.243, "apex": 2.094, "apex_t": 0.068, "landing_t": 0.118, "spin": 717.9, "composite": 0.128}
  },
  "Ablations of the final model": {
    "Final model (reference)": {"landing": 4.243, "apex": 2.094, "apex_t": 0.068, "landing_t": 0.118, "spin": 717.9, "composite": 0.128},
    "without session wind": {"landing": 4.507, "apex": 2.159, "apex_t": 0.077, "landing_t": 0.129, "spin": 717.9, "composite": 0.135},
    "without the physics feature (GP alone)": {"landing": 4.687, "apex": 2.350, "apex_t": 0.077, "landing_t": 0.130, "spin": 717.9, "composite": 0.138},
    "without fingerprint features": {"landing": 4.456, "apex": 2.296, "apex_t": 0.076, "landing_t": 0.125, "spin": 1008.9, "composite": 0.149},
    "with the true launch spin (oracle)": {"landing": 4.340, "apex": 2.110, "apex_t": 0.064, "landing_t": 0.117, "spin": 0.0, "composite": 0.093}
  }
}


In [ ]:
run("report.figures", "eda", tail=3)
run("report.figures", "results", tail=3)

from IPython.display import Image, display

for png in sorted((WORK / "figures").glob("*.png")):
    if png.name != "thumbnail.png":
        display(Image(filename=str(png)))

## 5. A test shot from launch to rest
Blue is what the radar tracks up to the net, orange is the modelled flight beyond it, and green is bounce and roll.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML

from report import style
from report.animation import animate, test_trajectories

style.apply()
matplotlib.rcParams["animation.embed_limit"] = 80
spins = pd.read_csv(WORK / "outputs" / "test_predictions_local.csv").set_index("track_id").launch_spin_rate
shot = [(spins - spins.quantile(0.2)).abs().idxmin()]
anim = animate(test_trajectories(shot), "A test shot: tracked to the net, predicted all the way to rest", fps=15)
try:
    player = anim.to_html5_video()
except Exception:  # no ffmpeg available: fall back to the JavaScript player
    player = anim.to_jshtml(default_mode="once")
plt.close(anim._fig)
HTML(player)

Optionally, render the GIFs used in the writeup: a single shot and a mix of five shots from low spin to wedge-like.

In [ ]:
run("report.animation", tail=3)
sorted(p.name for p in (WORK / "figures").glob("*.gif"))